# Topological Transition Example Figures

This notebook builds a manuscript-style multi-example figure for visually clear real regime transitions and their session-matched pseudo-transitions. Each example is shown as a `2 x 3` block: real versus pseudo on the rows, and pre-switch / peri-switch / post-switch windows on the columns.

Key plotting choices:
- Each example uses a joint PCA fitted to the real and pseudo windows together.
- Axis limits are shared across all six panels within an example block.
- The aspect ratio is fixed to `equal` so spatial extent is directly comparable.
- The color scale is fixed globally to relative time `-30` to `+20` s.
- SVG text is converted to paths so the exported figure preserves the intended Times-style typography.

In [7]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.decomposition import PCA

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

CWD = Path.cwd().resolve()
SRC_DIR = (CWD / 'src') if (CWD / 'src').exists() else (CWD.parent / 'src')
SRC_DIR = SRC_DIR.resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from affectdynamics.eval.transitions import _compute_run_labels
from affectdynamics.models.hmm import SharedEmissionHMM

plt.style.use('default')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
plt.rcParams['svg.fonttype'] = 'path'
plt.rcParams['axes.grid'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

REPO_ROOT = SRC_DIR.parent.resolve()
ARTIFACTS_DIR = REPO_ROOT / 'artifacts'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
CHMM_DIR = ARTIFACTS_DIR / '03_chmm_outputs' / 'chmm_8_full_dataset'
TRANSITIONS_DIR = ARTIFACTS_DIR / '07_transitions'
FIG_DIR = TRANSITIONS_DIR / 'notebook_figures'
TAB_DIR = TRANSITIONS_DIR / 'notebook_tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_JSON = CHMM_DIR / 'best_model.json'
MATCHED_ANCHORS_PATH = TRANSITIONS_DIR / 'matched_real_pseudo_anchors.csv'
SWITCH_SEGMENTS_PATH = TRANSITIONS_DIR / 'switch_local_segment_ph.csv'
MANIFEST_PATH = PROCESSED_DIR / 'processed_manifest.csv'

# Curated visually clear examples. Set to None to use the automatic fallback selector.
EXAMPLE_OVERRIDES = [
    {
        'session_id': '044_A',
        'switch_index': 3,
        'switch_time_idx': 80,
        'pseudo_switch_time_idx': 52,
        'transition_type': '0->1',
        'posterior_jump_l2': 1.3252472877502441,
        'segment_total_persistence_h1': 2.215505838394165,
        'regime_stability_type': 'persistent_to_persistent',
    },
    {
        'session_id': '121_A',
        'switch_index': 210,
        'switch_time_idx': 1536,
        'pseudo_switch_time_idx': 1524,
        'transition_type': '3->1',
        'posterior_jump_l2': 0.9263652563095093,
        'segment_total_persistence_h1': 0.9659498333930969,
        'regime_stability_type': 'persistent_to_persistent',
    },
    {
        'session_id': '144_A',
        'switch_index': 200,
        'switch_time_idx': 1858,
        'pseudo_switch_time_idx': 1845,
        'transition_type': '0->1',
        'posterior_jump_l2': 1.2888840436935425,
        'segment_total_persistence_h1': 0.9603590369224548,
        'regime_stability_type': 'persistent_to_persistent',
    },
]

DELAY_M = 3
DELAY_TAU = 1
TIME_NORM = Normalize(vmin=-30, vmax=20)
WINDOWS = [
    ('Pre-switch', -30, -5),
    ('Peri-switch', -5, 5),
    ('Post-switch', 5, 20),
]

def load_model(model_json: Path) -> SharedEmissionHMM:
    params = json.loads(model_json.read_text())
    model = SharedEmissionHMM(K=int(params['K']))
    model.pi = np.asarray(params['pi'], dtype=float)
    model.A = np.asarray(params['A'], dtype=float)
    model.pT = np.asarray(params['pT'], dtype=float)
    model.pC = np.asarray(params['pC'], dtype=float)
    return model

def load_session_paths(manifest_path: Path, processed_dir: Path) -> dict[str, Path]:
    manifest = pd.read_csv(manifest_path)
    return {str(row.session_id): processed_dir / row.file for row in manifest.itertuples()}

def posterior_matrix(model: SharedEmissionHMM, session_path: Path) -> np.ndarray:
    table = pq.read_table(session_path, columns=['T', 'C'])
    therapist = table['T'].to_numpy().astype(int, copy=False)
    client = table['C'].to_numpy().astype(int, copy=False)
    n = min(len(therapist), len(client))
    return model.posterior(therapist[:n], client[:n])

def build_delay_embedding(posterior: np.ndarray, m: int = 3, tau: int = 1) -> tuple[np.ndarray, np.ndarray]:
    span = (m - 1) * tau
    if len(posterior) <= span:
        return np.empty((0, posterior.shape[1] * m)), np.empty((0,), dtype=int)
    starts = np.arange(len(posterior) - span, dtype=int)[:, None]
    offsets = (np.arange(m, dtype=int) * tau)[None, :]
    idx = starts + offsets
    embedded = posterior[idx].reshape(len(starts), -1)
    centers = starts[:, 0] + span // 2
    return embedded, centers

def path_length(points: np.ndarray) -> float:
    if len(points) < 2:
        return 0.0
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())

def displacement(points: np.ndarray) -> float:
    if len(points) < 2:
        return 0.0
    return float(np.linalg.norm(points[-1] - points[0]))

def tortuosity(points: np.ndarray) -> float:
    disp = displacement(points)
    if disp <= 1e-8:
        return np.nan
    return path_length(points) / disp

def radius(points: np.ndarray) -> float:
    if len(points) < 2:
        return np.nan
    center = points.mean(axis=0)
    return float(np.mean(np.linalg.norm(points - center, axis=1)))

def choose_example_switches(n_examples: int = 3) -> list[dict]:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    matched = pd.read_csv(MATCHED_ANCHORS_PATH)
    run_labels = _compute_run_labels(CHMM_DIR)

    candidates = (
        switch_df
        .merge(run_labels, on=['session_id', 'switch_index'], how='left')
        .merge(
            matched,
            left_on=['session_id', 'switch_time_idx'],
            right_on=['session_id', 'real_switch_time_idx'],
            how='left',
        )
        .sort_values(['session_id', 'switch_index', 'norm_time_diff'])
        .drop_duplicates(['session_id', 'switch_index'])
        .dropna(subset=['pseudo_switch_time_idx'])
        .copy()
    )

    candidates['time_diff'] = (candidates['switch_time_idx'] - candidates['pseudo_switch_time_idx']).abs()
    candidates = candidates[
        (candidates['regime_stability_type'] == 'persistent_to_persistent')
        & (candidates['time_diff'] >= 10)
    ].copy()
    candidates['selection_score'] = (
        1.5 * candidates['time_diff'].rank(pct=True)
        + 1.0 * candidates['segment_total_persistence_h1'].rank(pct=True)
        + 1.0 * candidates['posterior_jump_l2'].rank(pct=True)
    )
    top = candidates.sort_values('selection_score', ascending=False).head(n_examples)
    return [
        {
            'session_id': str(row['session_id']),
            'switch_index': int(row['switch_index']),
            'switch_time_idx': int(row['switch_time_idx']),
            'pseudo_switch_time_idx': int(row['pseudo_switch_time_idx']),
            'transition_type': str(row['transition_type']),
            'posterior_jump_l2': float(row['posterior_jump_l2']),
            'segment_total_persistence_h1': float(row['segment_total_persistence_h1']),
            'regime_stability_type': str(row['regime_stability_type']),
        }
        for _, row in top.iterrows()
    ]

MODEL = load_model(MODEL_JSON)
SESSION_PATHS = load_session_paths(MANIFEST_PATH, PROCESSED_DIR)
EXAMPLES = [example.copy() for example in EXAMPLE_OVERRIDES] if EXAMPLE_OVERRIDES is not None else choose_example_switches(n_examples=3)
pd.DataFrame(EXAMPLES)


,session_id,switch_index,switch_time_idx,pseudo_switch_time_idx,transition_type,posterior_jump_l2,segment_total_persistence_h1,regime_stability_type
0,044_A,3,80,52,0->1,1.325247,2.215506,persistent_to_persistent
1,121_A,210,1536,1524,3->1,0.926365,0.965950,persistent_to_persistent
2,144_A,200,1858,1845,0->1,1.288884,0.960359,persistent_to_persistent


In [ ]:
def build_example_dataframe(example: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)

    real_mask = (centers >= example['switch_time_idx'] - 30) & (centers <= example['switch_time_idx'] + 30)
    pseudo_mask = (centers >= example['pseudo_switch_time_idx'] - 30) & (centers <= example['pseudo_switch_time_idx'] + 30)
    joint_embedding = np.vstack([embedded[real_mask], embedded[pseudo_mask]])
    pca = PCA(n_components=2)
    pca.fit(joint_embedding)

    rows = []
    for anchor_kind, anchor_idx, mask in [
        ('real', example['switch_time_idx'], real_mask),
        ('pseudo', example['pseudo_switch_time_idx'], pseudo_mask),
    ]:
        coords = pca.transform(embedded[mask])
        rel_times = centers[mask] - anchor_idx
        for pc_xy, rel_time in zip(coords, rel_times, strict=False):
            window_label = None
            for label, lo, hi in WINDOWS:
                if lo <= rel_time <= hi:
                    window_label = label
                    break
            if window_label is None:
                continue
            rows.append({
                'example_id': f"{example['session_id']}_switch{example['switch_index']}",
                'session_id': example['session_id'],
                'switch_index': example['switch_index'],
                'transition_type': example['transition_type'],
                'anchor_kind': anchor_kind,
                'anchor_time_idx': int(anchor_idx),
                'matched_time_diff_sec': abs(int(example['switch_time_idx']) - int(example['pseudo_switch_time_idx'])),
                'posterior_jump_l2': float(example['posterior_jump_l2']),
                'segment_total_persistence_h1': float(example['segment_total_persistence_h1']),
                'rel_time_sec': int(rel_time),
                'window_label': window_label,
                'pc1': float(pc_xy[0]),
                'pc2': float(pc_xy[1]),
            })

    df_plot = pd.DataFrame(rows)
    summary_rows = []
    for (example_id, anchor_kind, window_label), grp in df_plot.groupby(['example_id', 'anchor_kind', 'window_label']):
        pts = grp.sort_values('rel_time_sec')[['pc1', 'pc2']].to_numpy()
        base = grp.iloc[0]
        summary_rows.append({
            'example_id': example_id,
            'session_id': base['session_id'],
            'switch_index': int(base['switch_index']),
            'transition_type': base['transition_type'],
            'anchor_kind': anchor_kind,
            'window_label': window_label,
            'matched_time_diff_sec': int(base['matched_time_diff_sec']),
            'posterior_jump_l2': float(base['posterior_jump_l2']),
            'segment_total_persistence_h1': float(base['segment_total_persistence_h1']),
            'n_points': int(len(pts)),
            'path_length': path_length(pts),
            'displacement': displacement(pts),
            'tortuosity': tortuosity(pts),
            'radius': radius(pts),
        })
    df_summary = pd.DataFrame(summary_rows)
    return df_plot, df_summary

def style_axis(ax: plt.Axes) -> None:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.tick_params(axis='both', labelsize=9, width=0.8, length=3)

def plot_snapshot(ax: plt.Axes, df_sub: pd.DataFrame, title: str, xlim: tuple[float, float], ylim: tuple[float, float], show_ylabel: bool = False):
    ordered = df_sub.sort_values('rel_time_sec').copy()
    pts = ordered[['pc1', 'pc2']].to_numpy()
    rel = ordered['rel_time_sec'].to_numpy()
    sc = ax.scatter([], [])
    if len(pts) > 0:
        ax.plot(pts[:, 0], pts[:, 1], color='0.75', linewidth=1.0, zorder=0)
        sc = ax.scatter(
            pts[:, 0],
            pts[:, 1],
            c=rel,
            cmap='viridis',
            norm=TIME_NORM,
            s=28,
            edgecolors='white',
            linewidths=0.35,
            zorder=2,
        )
        ax.scatter(pts[0, 0], pts[0, 1], s=40, facecolor='white', edgecolor='black', linewidth=0.9, zorder=3)
        ax.scatter(pts[-1, 0], pts[-1, 1], s=40, facecolor='black', edgecolor='black', linewidth=0.9, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2' if show_ylabel else '', fontsize=10)
    style_axis(ax)
    return sc

plot_frames = []
summary_frames = []
for example in EXAMPLES:
    df_plot_i, df_summary_i = build_example_dataframe(example)
    plot_frames.append(df_plot_i)
    summary_frames.append(df_summary_i)

df_plot = pd.concat(plot_frames, ignore_index=True)
df_summary = pd.concat(summary_frames, ignore_index=True)
df_plot.to_csv(TAB_DIR / 'topo_transition_examples_points.csv', index=False)
df_summary.to_csv(TAB_DIR / 'topo_transition_examples_summary.csv', index=False)

fig, axes = plt.subplots(len(EXAMPLES) * 2, 3, figsize=(12.8, 4.6 * len(EXAMPLES)), squeeze=False)
fig.subplots_adjust(left=0.18, right=0.90, top=0.93, bottom=0.06, hspace=0.55, wspace=0.35)
window_order = [label for label, _, _ in WINDOWS]
last_scatter = None

for example_idx, example in enumerate(EXAMPLES):
    example_id = f"{example['session_id']}_switch{example['switch_index']}"
    df_example = df_plot[df_plot['example_id'] == example_id].copy()
    x_range = df_example['pc1'].max() - df_example['pc1'].min()
    y_range = df_example['pc2'].max() - df_example['pc2'].min()
    pad = 0.08 * max(x_range, y_range, 1e-6)
    xlim = (df_example['pc1'].min() - pad, df_example['pc1'].max() + pad)
    ylim = (df_example['pc2'].min() - pad, df_example['pc2'].max() + pad)

    for row_offset, anchor_kind in enumerate(['real', 'pseudo']):
        row_idx = 2 * example_idx + row_offset
        for col_idx, window_label in enumerate(window_order):
            ax = axes[row_idx, col_idx]
            sub = df_example[(df_example['anchor_kind'] == anchor_kind) & (df_example['window_label'] == window_label)].copy()
            last_scatter = plot_snapshot(
                ax,
                sub,
                window_label if row_idx == 0 else window_label,
                xlim=xlim,
                ylim=ylim,
                show_ylabel=(col_idx == 0),
            )
            if col_idx == 0:
                row_label = (
                    f"{example['session_id']} | {example['transition_type']} | Δt={abs(example['switch_time_idx'] - example['pseudo_switch_time_idx'])} s\n"
                    f"{'Real transition' if anchor_kind == 'real' else 'Matched pseudo-transition'}"
                )
                ax.text(
                    -0.52,
                    0.5,
                    row_label,
                    transform=ax.transAxes,
                    rotation=90,
                    va='center',
                    ha='center',
                    fontsize=9.8,
                )

        header_ax = axes[2 * example_idx, 1]
        header_ax.text(
            0.5,
            1.10,
            (
                f"Session {example['session_id']} | switch {example['switch_index']} | "
                f"jump={example['posterior_jump_l2']:.2f} | H1 persistence={example['segment_total_persistence_h1']:.2f}"
            ),
            transform=header_ax.transAxes,
            ha='center',
            va='bottom',
            fontsize=10.5,
        )

fig.suptitle(
    'Delay-Embedded Posterior Geometry Around Real and Pseudo Transitions',
    fontsize=15,
    y=0.995,
)
cbar = fig.colorbar(last_scatter, ax=axes, shrink=0.97, pad=0.02)
cbar.set_label('Relative time from anchor (s)', fontsize=11)
cbar.set_ticks([-30, -20, -10, 0, 10, 20])
cbar.ax.tick_params(labelsize=10)

fig_path = FIG_DIR / 'topo_transition_examples.svg'
png_path = FIG_DIR / 'topo_transition_examples.png'
fig.savefig(fig_path, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(png_path, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

display(pd.DataFrame(EXAMPLES))
display(df_summary.sort_values(['example_id', 'anchor_kind', 'window_label']).reset_index(drop=True))
print('Saved SVG to', fig_path)
print('Saved PNG to', png_path)


In [ ]:
from ripser import ripser

STORY_FIG_PATH = FIG_DIR / 'topo_transition_story_figure.svg'
STORY_PNG_PATH = FIG_DIR / 'topo_transition_story_figure.png'
STORY_POINTS_PATH = TAB_DIR / 'topo_transition_story_points.csv'
STORY_SELECTION_PATH = TAB_DIR / 'topo_transition_story_selection.csv'

# Curated story figure override. This is intentionally more visual than the auto-ranked choice.
STORY_OVERRIDE = {
    'session_id': '044_A',
    'switch_index': 3,
    'switch_time_idx': 80,
    'pseudo_switch_time_idx': 52,
    'transition_type': '0->1',
    'posterior_jump_l2': 1.3252472877502441,
    'segment_total_persistence_h1': 2.215505838394165,
    'regime_stability_type': 'persistent_to_persistent',
    'subset_name': 'all_8',
    'dims': list(range(8)),
    'projection_pair': (0, 1),
}

STATE_COLORS = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
    '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
]
TRAJ_NORM = Normalize(vmin=-30, vmax=30)

def max_h1_lifetime(diagram: np.ndarray) -> float:
    if diagram.size == 0:
        return 0.0
    finite = np.isfinite(diagram[:, 1])
    if not np.any(finite):
        return 0.0
    life = diagram[finite, 1] - diagram[finite, 0]
    life = life[life > 0]
    return float(life.max()) if life.size > 0 else 0.0

def persistence_diagram_h1(points_2d: np.ndarray) -> np.ndarray:
    if len(points_2d) < 4:
        return np.empty((0, 2), dtype=float)
    dgms = ripser(points_2d, maxdim=1)['dgms']
    return dgms[1] if len(dgms) > 1 else np.empty((0, 2), dtype=float)

def posterior_window_df(example: dict) -> pd.DataFrame:
    posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])
    rows = []
    for anchor_kind, anchor_idx in [('real', example['switch_time_idx']), ('pseudo', example['pseudo_switch_time_idx'])]:
        idx = np.arange(max(0, anchor_idx - 30), min(len(posterior), anchor_idx + 31), dtype=int)
        rel = idx - anchor_idx
        block = posterior[idx]
        for state in range(block.shape[1]):
            for t_rel, prob in zip(rel, block[:, state], strict=False):
                rows.append({
                    'session_id': example['session_id'],
                    'switch_index': example['switch_index'],
                    'transition_type': example['transition_type'],
                    'anchor_kind': anchor_kind,
                    'state': state,
                    'rel_time_sec': int(t_rel),
                    'probability': float(prob),
                })
    return pd.DataFrame(rows)

def build_projected_story_data(example: dict, dims: list[int], pair: tuple[int, int]) -> dict:
    posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])[:, dims]
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)
    real_mask = (centers >= example['switch_time_idx'] - 30) & (centers <= example['switch_time_idx'] + 30)
    pseudo_mask = (centers >= example['pseudo_switch_time_idx'] - 30) & (centers <= example['pseudo_switch_time_idx'] + 30)
    joint = np.vstack([embedded[real_mask], embedded[pseudo_mask]])
    n_components = min(3, joint.shape[1], joint.shape[0])
    pca = PCA(n_components=n_components)
    pca.fit(joint)

    rows = []
    pre_points = {}
    diagrams = {}
    for anchor_kind, anchor_idx, mask in [
        ('real', example['switch_time_idx'], real_mask),
        ('pseudo', example['pseudo_switch_time_idx'], pseudo_mask),
    ]:
        coords = pca.transform(embedded[mask])[:, list(pair)]
        rel_times = centers[mask] - anchor_idx
        for xy, rel_time in zip(coords, rel_times, strict=False):
            rows.append({
                'anchor_kind': anchor_kind,
                'rel_time_sec': int(rel_time),
                'x': float(xy[0]),
                'y': float(xy[1]),
            })
        pre = coords[(rel_times >= -30) & (rel_times <= -5)]
        pre_points[anchor_kind] = pre
        diagrams[anchor_kind] = persistence_diagram_h1(pre)

    df_traj = pd.DataFrame(rows)
    return {
        'df_traj': df_traj,
        'pre_points': pre_points,
        'diagrams': diagrams,
        'explained_variance_ratio': pca.explained_variance_ratio_.tolist(),
    }

def choose_story_example(candidate_examples: list[dict]) -> tuple[dict, dict, pd.DataFrame]:
    candidate_rows = []
    best_payload = None
    best_example = None
    best_score = -np.inf

    for example in candidate_examples:
        posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])
        window_real = posterior[max(0, example['switch_time_idx'] - 30): example['switch_time_idx'] - 4]
        window_pseudo = posterior[max(0, example['pseudo_switch_time_idx'] - 30): example['pseudo_switch_time_idx'] - 4]
        if len(window_real) < 10 or len(window_pseudo) < 10:
            continue
        dim_rank = np.argsort(-np.abs(window_real.mean(axis=0) - window_pseudo.mean(axis=0)))
        subsets = [
            ('all_8', list(range(posterior.shape[1]))),
            ('top_3', sorted(dim_rank[:3].tolist())),
            ('top_2', sorted(dim_rank[:2].tolist())),
        ]

        for subset_name, dims in subsets:
            payload = build_projected_story_data(example, dims=dims, pair=(0, 1))
            n_comp = len(payload['explained_variance_ratio'])
            pairs = [(0, 1)] if n_comp < 3 else [(0, 1), (0, 2), (1, 2)]
            for pair in pairs:
                payload = build_projected_story_data(example, dims=dims, pair=pair)
                df_traj = payload['df_traj']
                real_pre = payload['pre_points']['real']
                pseudo_pre = payload['pre_points']['pseudo']
                real_h1 = max_h1_lifetime(payload['diagrams']['real'])
                pseudo_h1 = max_h1_lifetime(payload['diagrams']['pseudo'])
                real_pre_tort = tortuosity(real_pre)
                pseudo_pre_tort = tortuosity(pseudo_pre)
                real_post = df_traj[(df_traj['anchor_kind'] == 'real') & (df_traj['rel_time_sec'].between(5, 20))][['x', 'y']].to_numpy()
                pseudo_post = df_traj[(df_traj['anchor_kind'] == 'pseudo') & (df_traj['rel_time_sec'].between(5, 20))][['x', 'y']].to_numpy()
                real_post_radius = radius(real_post)
                pseudo_post_radius = radius(pseudo_post)
                score = (
                    2.5 * (real_h1 - pseudo_h1)
                    + 0.4 * np.nan_to_num(real_pre_tort - pseudo_pre_tort, nan=0.0)
                    + 0.3 * np.nan_to_num(radius(real_pre) - real_post_radius, nan=0.0)
                    - 0.2 * np.nan_to_num(pseudo_post_radius, nan=0.0)
                )
                row = {
                    'session_id': example['session_id'],
                    'switch_index': example['switch_index'],
                    'transition_type': example['transition_type'],
                    'time_diff_sec': abs(example['switch_time_idx'] - example['pseudo_switch_time_idx']),
                    'subset_name': subset_name,
                    'dims': ','.join(map(str, dims)),
                    'projection_pair': f'PC{pair[0] + 1}-PC{pair[1] + 1}',
                    'real_h1_max_lifetime': real_h1,
                    'pseudo_h1_max_lifetime': pseudo_h1,
                    'real_pre_tortuosity': real_pre_tort,
                    'pseudo_pre_tortuosity': pseudo_pre_tort,
                    'real_post_radius': real_post_radius,
                    'pseudo_post_radius': pseudo_post_radius,
                    'score': score,
                }
                candidate_rows.append(row)
                if score > best_score:
                    best_score = score
                    best_payload = payload | {
                        'subset_name': subset_name,
                        'dims': dims,
                        'projection_pair': pair,
                    }
                    best_example = example

    df_candidates = pd.DataFrame(candidate_rows).sort_values('score', ascending=False).reset_index(drop=True)
    best_row = df_candidates.iloc[0].to_dict()
    best_payload = build_projected_story_data(best_example, dims=best_payload['dims'], pair=best_payload['projection_pair']) | {
        'subset_name': best_payload['subset_name'],
        'dims': best_payload['dims'],
        'projection_pair': best_payload['projection_pair'],
    }
    return best_example, best_payload, df_candidates

story_example, story_payload, df_story_candidates = choose_story_example(EXAMPLES)
if STORY_OVERRIDE is not None:
    story_example = {k: STORY_OVERRIDE[k] for k in ['session_id', 'switch_index', 'switch_time_idx', 'pseudo_switch_time_idx', 'transition_type', 'posterior_jump_l2', 'segment_total_persistence_h1', 'regime_stability_type']}
    story_payload = build_projected_story_data(story_example, dims=STORY_OVERRIDE['dims'], pair=STORY_OVERRIDE['projection_pair']) | {
        'subset_name': STORY_OVERRIDE['subset_name'],
        'dims': STORY_OVERRIDE['dims'],
        'projection_pair': STORY_OVERRIDE['projection_pair'],
    }
df_story_candidates.to_csv(STORY_SELECTION_PATH, index=False)

df_story_probs = posterior_window_df(story_example)
df_story_traj = story_payload['df_traj'].copy()
df_story_traj['session_id'] = story_example['session_id']
df_story_traj['switch_index'] = story_example['switch_index']
df_story_traj['transition_type'] = story_example['transition_type']
df_story_traj['subset_name'] = story_payload['subset_name']
df_story_traj['dims'] = ','.join(map(str, story_payload['dims']))
df_story_traj['projection_pair'] = f"PC{story_payload['projection_pair'][0] + 1}-PC{story_payload['projection_pair'][1] + 1}"
df_story_traj.to_csv(STORY_POINTS_PATH, index=False)

fig, axes = plt.subplots(3, 2, figsize=(11.5, 11.0), constrained_layout=True)

# Row 1: raw posterior probabilities.
for col_idx, anchor_kind in enumerate(['real', 'pseudo']):
    ax = axes[0, col_idx]
    sub = df_story_probs[df_story_probs['anchor_kind'] == anchor_kind].copy()
    for state in range(8):
        s = sub[sub['state'] == state].sort_values('rel_time_sec')
        ax.plot(s['rel_time_sec'], s['probability'], color=STATE_COLORS[state], linewidth=1.4, alpha=0.95)
    ax.axvline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.7)
    ax.set_xlim(-30, 30)
    ax.set_ylim(0, 1)
    ax.set_title(f"{'Real transition' if anchor_kind == 'real' else 'Matched pseudo-transition'}", fontsize=12)
    ax.set_xlabel('Relative time from anchor (s)', fontsize=10)
    ax.set_ylabel('Posterior probability', fontsize=10)
    style_axis(ax)

# Row 2: projected delay-embedded geometry.
x_range = df_story_traj['x'].max() - df_story_traj['x'].min()
y_range = df_story_traj['y'].max() - df_story_traj['y'].min()
pad = 0.08 * max(x_range, y_range, 1e-6)
story_xlim = (df_story_traj['x'].min() - pad, df_story_traj['x'].max() + pad)
story_ylim = (df_story_traj['y'].min() - pad, df_story_traj['y'].max() + pad)

for col_idx, anchor_kind in enumerate(['real', 'pseudo']):
    ax = axes[1, col_idx]
    sub = df_story_traj[df_story_traj['anchor_kind'] == anchor_kind].sort_values('rel_time_sec')
    pts = sub[['x', 'y']].to_numpy()
    rel = sub['rel_time_sec'].to_numpy()
    ax.plot(pts[:, 0], pts[:, 1], color='0.75', linewidth=1.1, zorder=0)
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=rel, cmap='viridis', norm=TRAJ_NORM, s=34, edgecolors='white', linewidths=0.35, zorder=2)
    ax.scatter(pts[0, 0], pts[0, 1], s=44, facecolor='white', edgecolor='black', linewidth=0.9, zorder=3)
    ax.scatter(pts[-1, 0], pts[-1, 1], s=44, facecolor='black', edgecolor='black', linewidth=0.9, zorder=3)
    ax.set_xlim(story_xlim)
    ax.set_ylim(story_ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(f"{df_story_traj['projection_pair'].iloc[0].split('-')[0]}", fontsize=10)
    ax.set_ylabel(f"{df_story_traj['projection_pair'].iloc[0].split('-')[1]}", fontsize=10)
    style_axis(ax)

# Row 3: persistence diagrams for the pre-switch point clouds.
real_dgm = story_payload['diagrams']['real']
pseudo_dgm = story_payload['diagrams']['pseudo']
all_vals = []
for dgm in [real_dgm, pseudo_dgm]:
    if dgm.size:
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite):
            all_vals.extend(finite[:, 0].tolist())
            all_vals.extend(finite[:, 1].tolist())
diag_max = max(all_vals) if all_vals else 1.0
diag_max = float(diag_max) * 1.05 if diag_max > 0 else 1.0

for col_idx, (anchor_kind, dgm) in enumerate([('real', real_dgm), ('pseudo', pseudo_dgm)]):
    ax = axes[2, col_idx]
    ax.plot([0, diag_max], [0, diag_max], color='0.55', linestyle='--', linewidth=1.0)
    finite = dgm[np.isfinite(dgm[:, 1])] if dgm.size else np.empty((0, 2))
    if len(finite):
        ax.scatter(finite[:, 0], finite[:, 1], s=46, color='#1f77b4', edgecolors='white', linewidths=0.4)
    else:
        ax.text(0.5, 0.52, 'No H1 feature', ha='center', va='center', transform=ax.transAxes, fontsize=10)
    ax.set_xlim(0, diag_max)
    ax.set_ylim(0, diag_max)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Birth', fontsize=10)
    ax.set_ylabel('Death', fontsize=10)
    style_axis(ax)

fig.suptitle('Same Posterior Signal, Different Geometry, Captured by Topology', fontsize=15, y=1.01)
axes[0, 0].text(-0.40, 0.5, 'Row 1\nRaw posterior\nprobabilities', transform=axes[0, 0].transAxes, rotation=90, va='center', ha='center', fontsize=10.5)
axes[1, 0].text(-0.40, 0.5, 'Row 2\nDelay-embedded\ntrajectory', transform=axes[1, 0].transAxes, rotation=90, va='center', ha='center', fontsize=10.5)
axes[2, 0].text(-0.40, 0.5, 'Row 3\nPre-switch\npersistence diagram', transform=axes[2, 0].transAxes, rotation=90, va='center', ha='center', fontsize=10.5)

fig.text(
    0.5,
    0.965,
    (
        f"Selected example: session {story_example['session_id']} | switch {story_example['switch_index']} | "
        f"transition {story_example['transition_type']} | pseudo offset {abs(story_example['switch_time_idx'] - story_example['pseudo_switch_time_idx'])} s | "
        f"projection {df_story_traj['projection_pair'].iloc[0]} on {df_story_traj['subset_name'].iloc[0]} dimensions ({df_story_traj['dims'].iloc[0]})"
    ),
    ha='center',
    fontsize=10.5,
)
cbar = fig.colorbar(sc, ax=axes[1, :], shrink=0.95, pad=0.02)
cbar.set_label('Relative time from anchor (s)', fontsize=10)
cbar.set_ticks([-30, -20, -10, 0, 10, 20, 30])
cbar.ax.tick_params(labelsize=9)

fig.savefig(STORY_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(STORY_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

display(df_story_candidates.head(15))
print('Selected story example:', story_example)
print('Projection subset:', story_payload['subset_name'], story_payload['dims'])
print('Projection pair:', f"PC{story_payload['projection_pair'][0] + 1}-PC{story_payload['projection_pair'][1] + 1}")
print('Saved SVG to', STORY_FIG_PATH)
print('Saved PNG to', STORY_PNG_PATH)


In [ ]:
REAL_ONLY_FIG_PATH = FIG_DIR / 'topo_transition_pre_peri_post_real_only.svg'
REAL_ONLY_PNG_PATH = FIG_DIR / 'topo_transition_pre_peri_post_real_only.png'
REAL_ONLY_POINTS_PATH = TAB_DIR / 'topo_transition_pre_peri_post_real_only_points.csv'
REAL_ONLY_DIAGRAMS_PATH = TAB_DIR / 'topo_transition_pre_peri_post_real_only_diagrams.csv'
REAL_ONLY_SELECTION_PATH = TAB_DIR / 'topo_transition_pre_peri_post_real_only_selection.csv'

REAL_ONLY_OVERRIDE = None

def real_only_projection_payload(example: dict, dims: list[int], pair: tuple[int, int]) -> dict:
    posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])[:, dims]
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)
    mask = (centers >= example['switch_time_idx'] - 30) & (centers <= example['switch_time_idx'] + 20)
    selected = embedded[mask]
    if len(selected) < 10:
        raise ValueError('Not enough embedded points for real-only figure.')
    n_components = min(3, selected.shape[0], selected.shape[1])
    projected_full = PCA(n_components=n_components).fit_transform(selected)
    if max(pair) >= projected_full.shape[1]:
        raise ValueError('Projection pair exceeds available PCA dimensions.')
    projected = projected_full[:, list(pair)]
    rel_times = centers[mask] - example['switch_time_idx']

    point_rows = []
    diagram_rows = []
    metrics = {}
    for window_label, lo, hi in WINDOWS:
        keep = (rel_times >= lo) & (rel_times <= hi)
        pts = projected[keep]
        rel = rel_times[keep]
        for xy, rel_t in zip(pts, rel, strict=False):
            point_rows.append({
                'session_id': example['session_id'],
                'switch_index': example['switch_index'],
                'transition_type': example['transition_type'],
                'subset_name': example['subset_name'],
                'projection_pair': f"PC{pair[0] + 1}-PC{pair[1] + 1}",
                'window_label': window_label,
                'rel_time_sec': int(rel_t),
                'x': float(xy[0]),
                'y': float(xy[1]),
            })
        dgm = persistence_diagram_h1(pts)
        finite = dgm[np.isfinite(dgm[:, 1])] if dgm.size else np.empty((0, 2), dtype=float)
        metrics[window_label] = {
            'max_h1': max_h1_lifetime(finite),
            'radius': radius(pts),
            'tortuosity': tortuosity(pts),
        }
        if len(finite) == 0:
            diagram_rows.append({
                'window_label': window_label,
                'birth': np.nan,
                'death': np.nan,
                'lifetime': np.nan,
            })
        else:
            for birth, death in finite:
                diagram_rows.append({
                    'window_label': window_label,
                    'birth': float(birth),
                    'death': float(death),
                    'lifetime': float(death - birth),
                })
    return {
        'df_points': pd.DataFrame(point_rows),
        'df_diagrams': pd.DataFrame(diagram_rows),
        'metrics': metrics,
    }

def choose_real_only_example() -> tuple[dict, dict, pd.DataFrame]:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    run_labels = _compute_run_labels(CHMM_DIR)
    candidates = switch_df.merge(run_labels, on=['session_id', 'switch_index'], how='left').copy()
    candidates = candidates[candidates['regime_stability_type'].eq('persistent_to_persistent')].copy()
    candidates = candidates.sort_values(['segment_total_persistence_h1', 'posterior_jump_l2'], ascending=False).head(250)

    selection_rows = []
    best_score = -np.inf
    best_example = None
    best_payload = None

    for row in candidates.itertuples():
        example = {
            'session_id': str(row.session_id),
            'switch_index': int(row.switch_index),
            'switch_time_idx': int(row.switch_time_idx),
            'transition_type': str(row.transition_type),
            'posterior_jump_l2': float(row.posterior_jump_l2),
            'segment_total_persistence_h1': float(row.segment_total_persistence_h1),
        }
        try:
            posterior = posterior_matrix(MODEL, SESSION_PATHS[example['session_id']])
        except Exception:
            continue
        if example['switch_time_idx'] < 35 or example['switch_time_idx'] > len(posterior) - 25:
            continue

        local = posterior[max(0, example['switch_time_idx'] - 30): min(len(posterior), example['switch_time_idx'] + 21)]
        var_rank = np.argsort(-local.var(axis=0))
        subsets = [
            ('all_8', list(range(posterior.shape[1]))),
            ('top_3_var', sorted(var_rank[:3].tolist())),
            ('top_2_var', sorted(var_rank[:2].tolist())),
        ]

        for subset_name, dims in subsets:
            tmp = example | {'subset_name': subset_name, 'dims': dims}
            for pair in [(0, 1), (0, 2), (1, 2)]:
                try:
                    payload = real_only_projection_payload(tmp, dims=dims, pair=pair)
                except Exception:
                    continue
                pre = payload['metrics']['Pre-switch']
                peri = payload['metrics']['Peri-switch']
                post = payload['metrics']['Post-switch']
                score = (
                    1.6 * pre['max_h1']
                    + 1.8 * peri['max_h1']
                    - 2.4 * post['max_h1']
                    + 0.9 * np.nan_to_num(pre['radius'], nan=0.0)
                    + 0.9 * np.nan_to_num(peri['radius'], nan=0.0)
                    - 1.6 * np.nan_to_num(post['radius'], nan=0.0)
                    + 0.25 * np.nan_to_num(pre['tortuosity'], nan=0.0)
                    + 0.25 * np.nan_to_num(peri['tortuosity'], nan=0.0)
                    - 0.10 * np.nan_to_num(post['tortuosity'], nan=0.0)
                )
                selection_rows.append({
                    'session_id': example['session_id'],
                    'switch_index': example['switch_index'],
                    'switch_time_idx': example['switch_time_idx'],
                    'transition_type': example['transition_type'],
                    'posterior_jump_l2': example['posterior_jump_l2'],
                    'segment_total_persistence_h1': example['segment_total_persistence_h1'],
                    'subset_name': subset_name,
                    'dims': ','.join(map(str, dims)),
                    'projection_pair': f"PC{pair[0] + 1}-PC{pair[1] + 1}",
                    'pre_h1': pre['max_h1'],
                    'peri_h1': peri['max_h1'],
                    'post_h1': post['max_h1'],
                    'pre_radius': pre['radius'],
                    'peri_radius': peri['radius'],
                    'post_radius': post['radius'],
                    'pre_tortuosity': pre['tortuosity'],
                    'peri_tortuosity': peri['tortuosity'],
                    'post_tortuosity': post['tortuosity'],
                    'score': score,
                })
                if score > best_score:
                    best_score = score
                    best_example = tmp | {'projection_pair': pair}
                    best_payload = payload

    df_selection = pd.DataFrame(selection_rows).sort_values('score', ascending=False).reset_index(drop=True)
    return best_example, best_payload, df_selection

REAL_ONLY_EXAMPLE, real_only_payload, df_real_only_selection = choose_real_only_example()
if REAL_ONLY_OVERRIDE is not None:
    REAL_ONLY_EXAMPLE = REAL_ONLY_OVERRIDE.copy()
    real_only_payload = real_only_projection_payload(REAL_ONLY_EXAMPLE, dims=REAL_ONLY_EXAMPLE['dims'], pair=REAL_ONLY_EXAMPLE['projection_pair'])
df_real_only_selection.to_csv(REAL_ONLY_SELECTION_PATH, index=False)

df_real_only_points = real_only_payload['df_points'].copy()
df_real_only_diagrams = real_only_payload['df_diagrams'].copy()
df_real_only_points.to_csv(REAL_ONLY_POINTS_PATH, index=False)
df_real_only_diagrams.to_csv(REAL_ONLY_DIAGRAMS_PATH, index=False)

fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.8), constrained_layout=True)
x_range = df_real_only_points['x'].max() - df_real_only_points['x'].min()
y_range = df_real_only_points['y'].max() - df_real_only_points['y'].min()
pad = 0.08 * max(x_range, y_range, 1e-6)
xlim = (df_real_only_points['x'].min() - pad, df_real_only_points['x'].max() + pad)
ylim = (df_real_only_points['y'].min() - pad, df_real_only_points['y'].max() + pad)

diag_vals = df_real_only_diagrams[['birth', 'death']].to_numpy().ravel()
diag_vals = diag_vals[np.isfinite(diag_vals)]
diag_max = float(diag_vals.max()) * 1.05 if diag_vals.size else 1.0

for col_idx, (window_label, lo, hi) in enumerate(WINDOWS):
    ax = axes[0, col_idx]
    sub = df_real_only_points[df_real_only_points['window_label'] == window_label].sort_values('rel_time_sec')
    pts = sub[['x', 'y']].to_numpy()
    rel = sub['rel_time_sec'].to_numpy()
    ax.plot(pts[:, 0], pts[:, 1], color='0.75', linewidth=1.1, zorder=0)
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=rel, cmap='viridis', norm=Normalize(vmin=-30, vmax=20), s=34, edgecolors='white', linewidths=0.35, zorder=2)
    ax.scatter(pts[0, 0], pts[0, 1], s=44, facecolor='white', edgecolor='black', linewidth=0.9, zorder=3)
    ax.scatter(pts[-1, 0], pts[-1, 1], s=44, facecolor='black', edgecolor='black', linewidth=0.9, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(window_label, fontsize=12)
    ax.set_xlabel(f"PC{REAL_ONLY_EXAMPLE['projection_pair'][0] + 1}", fontsize=10)
    ax.set_ylabel(f"PC{REAL_ONLY_EXAMPLE['projection_pair'][1] + 1}", fontsize=10)
    style_axis(ax)

    ax_d = axes[1, col_idx]
    ax_d.plot([0, diag_max], [0, diag_max], color='0.55', linestyle='--', linewidth=1.0)
    dsub = df_real_only_diagrams[(df_real_only_diagrams['window_label'] == window_label) & df_real_only_diagrams['birth'].notna()].copy()
    if dsub.empty:
        ax_d.text(0.5, 0.52, 'No H1 feature', ha='center', va='center', transform=ax_d.transAxes, fontsize=10)
    else:
        ax_d.scatter(dsub['birth'], dsub['death'], s=48, color='#1f77b4', edgecolors='white', linewidths=0.4)
        top = dsub.sort_values('lifetime', ascending=False).iloc[0]
        ax_d.text(top['birth'], top['death'], f"  H1={top['lifetime']:.2f}", fontsize=9, va='bottom')
    ax_d.set_xlim(0, diag_max)
    ax_d.set_ylim(0, diag_max)
    ax_d.set_aspect('equal', adjustable='box')
    ax_d.set_xlabel('Birth', fontsize=10)
    ax_d.set_ylabel('Death', fontsize=10)
    style_axis(ax_d)

fig.suptitle('Pre-switch Winding and Post-switch Collapse in a Single Real Transition', fontsize=15, y=1.02)
fig.text(
    0.5,
    0.965,
    (
        f"Session {REAL_ONLY_EXAMPLE['session_id']} | switch {REAL_ONLY_EXAMPLE['switch_index']} | transition {REAL_ONLY_EXAMPLE['transition_type']} | "
        f"jump={REAL_ONLY_EXAMPLE['posterior_jump_l2']:.2f} | H1 persistence={REAL_ONLY_EXAMPLE['segment_total_persistence_h1']:.2f}"
    ),
    ha='center',
    fontsize=10.5,
)
axes[0, 0].text(-0.38, 0.5, 'Delay-embedded\ntrajectory', transform=axes[0, 0].transAxes, rotation=90, va='center', ha='center', fontsize=10.5)
axes[1, 0].text(-0.38, 0.5, 'H1 persistence\ndiagram', transform=axes[1, 0].transAxes, rotation=90, va='center', ha='center', fontsize=10.5)
cbar = fig.colorbar(sc, ax=axes[0, :], shrink=0.95, pad=0.02)
cbar.set_label('Relative time from switch (s)', fontsize=10)
cbar.set_ticks([-30, -20, -10, 0, 10, 20])
cbar.ax.tick_params(labelsize=9)

fig.savefig(REAL_ONLY_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(REAL_ONLY_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

display(df_real_only_diagrams.sort_values(['window_label', 'lifetime'], ascending=[True, False]).reset_index(drop=True))
print('Saved SVG to', REAL_ONLY_FIG_PATH)
print('Saved PNG to', REAL_ONLY_PNG_PATH)


In [ ]:
from ripser import ripser

AGG_FIG_PATH = FIG_DIR / 'topo_transition_aggregate_tda_summaries.svg'
AGG_PNG_PATH = FIG_DIR / 'topo_transition_aggregate_tda_summaries.png'
AGG_LANDSCAPE_PATH = TAB_DIR / 'topo_transition_aggregate_landscape.csv'
AGG_CLOUD_PATH = TAB_DIR / 'topo_transition_aggregate_birth_death_cloud.csv'
AGG_BETTI_PATH = TAB_DIR / 'topo_transition_aggregate_betti_curves.csv'

AGGREGATE_SAMPLE_SIZE = 300
AGGREGATE_RANDOM_SEED = 7

def get_anchor_table() -> pd.DataFrame:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    matched = pd.read_csv(MATCHED_ANCHORS_PATH)
    pseudo_map = (
        matched
        .sort_values(['session_id', 'real_switch_time_idx', 'norm_time_diff'])
        .drop_duplicates(['session_id', 'real_switch_time_idx'])
        .rename(columns={'real_switch_time_idx': 'switch_time_idx'})
    )
    anchors = (
        switch_df
        .merge(
            pseudo_map[['session_id', 'switch_time_idx', 'pseudo_switch_time_idx', 'norm_time_diff']],
            on=['session_id', 'switch_time_idx'],
            how='inner',
        )
        .drop_duplicates(['session_id', 'switch_index'])
        .copy()
    )
    return anchors

def sample_anchor_table(df: pd.DataFrame, n: int | None, seed: int = 7) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).sort_values(['session_id', 'switch_index']).reset_index(drop=True)

def h1_diagram_for_window(session_id: str, anchor_idx: int, lo: int, hi: int, dims: list[int] | None = None) -> np.ndarray:
    posterior = posterior_matrix(MODEL, SESSION_PATHS[session_id])
    if dims is not None:
        posterior = posterior[:, dims]
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)
    keep = (centers >= anchor_idx + lo) & (centers <= anchor_idx + hi)
    pts = embedded[keep]
    if len(pts) < 4:
        return np.empty((0, 2), dtype=float)
    dgms = ripser(pts, maxdim=1)['dgms']
    dgm = dgms[1] if len(dgms) > 1 else np.empty((0, 2), dtype=float)
    return dgm[np.isfinite(dgm[:, 1])] if dgm.size else np.empty((0, 2), dtype=float)

def persistence_landscape(diagram: np.ndarray, grid: np.ndarray, n_layers: int = 3) -> np.ndarray:
    if diagram.size == 0:
        return np.zeros((n_layers, len(grid)), dtype=float)
    tents = []
    for birth, death in diagram:
        tents.append(np.maximum(0.0, np.minimum(grid - birth, death - grid)))
    tent_array = np.vstack(tents) if tents else np.zeros((0, len(grid)))
    sorted_vals = np.sort(tent_array, axis=0)[::-1] if tent_array.size else np.zeros((0, len(grid)))
    out = np.zeros((n_layers, len(grid)), dtype=float)
    for layer in range(n_layers):
        if layer < sorted_vals.shape[0]:
            out[layer] = sorted_vals[layer]
    return out

def betti_curve(diagram: np.ndarray, grid: np.ndarray) -> np.ndarray:
    if diagram.size == 0:
        return np.zeros(len(grid), dtype=float)
    births = diagram[:, 0][:, None]
    deaths = diagram[:, 1][:, None]
    eps = grid[None, :]
    return ((births <= eps) & (eps < deaths)).sum(axis=0).astype(float)

anchor_table = sample_anchor_table(get_anchor_table(), AGGREGATE_SAMPLE_SIZE, AGGREGATE_RANDOM_SEED)

real_pre_diagrams = []
pseudo_pre_diagrams = []
real_post_diagrams = []
for row in anchor_table.itertuples():
    real_pre_diagrams.append(h1_diagram_for_window(str(row.session_id), int(row.switch_time_idx), -30, -5))
    pseudo_pre_diagrams.append(h1_diagram_for_window(str(row.session_id), int(row.pseudo_switch_time_idx), -30, -5))
    real_post_diagrams.append(h1_diagram_for_window(str(row.session_id), int(row.switch_time_idx), 5, 20))

cloud_rows = []
for group, diagrams in [('real_pre', real_pre_diagrams), ('pseudo_pre', pseudo_pre_diagrams), ('real_post', real_post_diagrams)]:
    for idx, diag in enumerate(diagrams):
        for birth, death in diag:
            cloud_rows.append({'group': group, 'diagram_index': idx, 'birth': float(birth), 'death': float(death), 'lifetime': float(death - birth)})
df_cloud = pd.DataFrame(cloud_rows)
df_cloud.to_csv(AGG_CLOUD_PATH, index=False)

grid_max = max(df_cloud['death'].max() if not df_cloud.empty else 1.0, 1.0)
grid = np.linspace(0.0, float(grid_max) * 1.05, 200)

landscape_rows = []
for group, diagrams in [('real_pre', real_pre_diagrams), ('pseudo_pre', pseudo_pre_diagrams)]:
    landscapes = np.stack([persistence_landscape(diag, grid, n_layers=3) for diag in diagrams], axis=0)
    mean_landscape = landscapes.mean(axis=0)
    for layer in range(mean_landscape.shape[0]):
        for x, y in zip(grid, mean_landscape[layer], strict=False):
            landscape_rows.append({'group': group, 'layer': layer + 1, 'filtration_value': float(x), 'landscape_value': float(y)})
df_landscape = pd.DataFrame(landscape_rows)
df_landscape.to_csv(AGG_LANDSCAPE_PATH, index=False)

betti_rows = []
for group, diagrams in [('real_pre', real_pre_diagrams), ('real_post', real_post_diagrams)]:
    curves = np.stack([betti_curve(diag, grid) for diag in diagrams], axis=0)
    mean_curve = curves.mean(axis=0)
    for x, y in zip(grid, mean_curve, strict=False):
        betti_rows.append({'group': group, 'filtration_value': float(x), 'mean_beta_1': float(y)})
df_betti = pd.DataFrame(betti_rows)
df_betti.to_csv(AGG_BETTI_PATH, index=False)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.3), constrained_layout=True)

for group, color in [('real_pre', '#1f77b4'), ('pseudo_pre', '#d62728')]:
    sub = df_landscape[(df_landscape['group'] == group) & (df_landscape['layer'] == 1)].copy()
    axes[0].plot(sub['filtration_value'], sub['landscape_value'], color=color, linewidth=2.0, label='Real pre-switch' if group == 'real_pre' else 'Pseudo pre-switch')
axes[0].set_title('Mean Persistence Landscape (Layer 1)', fontsize=12)
axes[0].set_xlabel('Filtration value', fontsize=10)
axes[0].set_ylabel('Landscape value', fontsize=10)
axes[0].legend(frameon=False, fontsize=9)
style_axis(axes[0])

if not df_cloud.empty:
    for group, color, alpha in [('real_pre', '#1f77b4', 0.45), ('pseudo_pre', '#d62728', 0.35)]:
        sub = df_cloud[df_cloud['group'] == group].copy()
        axes[1].scatter(sub['birth'], sub['death'], s=18, color=color, alpha=alpha, label='Real pre-switch' if group == 'real_pre' else 'Pseudo pre-switch')
diag_max = max(float(grid.max()), 1.0)
axes[1].plot([0, diag_max], [0, diag_max], linestyle='--', color='0.6', linewidth=1.0)
axes[1].set_xlim(0, diag_max)
axes[1].set_ylim(0, diag_max)
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_title('H1 Birth-Death Cloud', fontsize=12)
axes[1].set_xlabel('Birth', fontsize=10)
axes[1].set_ylabel('Death', fontsize=10)
axes[1].legend(frameon=False, fontsize=9, loc='lower right')
style_axis(axes[1])

for group, color in [('real_pre', '#1f77b4'), ('real_post', '#2ca02c')]:
    sub = df_betti[df_betti['group'] == group].copy()
    axes[2].plot(sub['filtration_value'], sub['mean_beta_1'], color=color, linewidth=2.0, label='Real pre-switch' if group == 'real_pre' else 'Real post-switch')
axes[2].set_title('Mean Betti Curve', fontsize=12)
axes[2].set_xlabel('Filtration value', fontsize=10)
axes[2].set_ylabel(r'Mean $\beta_1(\epsilon)$', fontsize=10)
axes[2].legend(frameon=False, fontsize=9)
style_axis(axes[2])

fig.suptitle(f'Aggregate TDA Summaries Across {len(anchor_table)} Matched Transition Anchors', fontsize=14, y=1.03)
fig.savefig(AGG_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(AGG_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Aggregate anchors used:', len(anchor_table))
print('Saved SVG to', AGG_FIG_PATH)
print('Saved PNG to', AGG_PNG_PATH)


In [ ]:
from ripser import ripser

BETTI_ARC_FIG_PATH = FIG_DIR / 'topo_transition_real_betti_arc.svg'
BETTI_ARC_PNG_PATH = FIG_DIR / 'topo_transition_real_betti_arc.png'
BETTI_ARC_CURVES_PATH = TAB_DIR / 'topo_transition_real_betti_arc_curves.csv'
BETTI_ARC_SUMMARY_PATH = TAB_DIR / 'topo_transition_real_betti_arc_summary.csv'

BETTI_ARC_CENTERS = [-25, -15, -5, 5, 15, 25]
BETTI_ARC_HALF_WIDTH = 5
BETTI_ARC_SAMPLE_SIZE = 300
BETTI_ARC_RANDOM_SEED = 11
BETTI_ARC_GRID_SIZE = 180
BETTI_ARC_SMOOTH_SIGMA = 1.6

POSTERIOR_CACHE: dict[str, np.ndarray] = {}

def get_cached_posterior(session_id: str) -> np.ndarray:
    if session_id not in POSTERIOR_CACHE:
        POSTERIOR_CACHE[session_id] = posterior_matrix(MODEL, SESSION_PATHS[session_id])
    return POSTERIOR_CACHE[session_id]

def betti_curve_from_diagram(diagram: np.ndarray, grid: np.ndarray) -> np.ndarray:
    if diagram.size == 0:
        return np.zeros_like(grid)
    births = diagram[:, 0][:, None]
    deaths = diagram[:, 1][:, None]
    return ((births <= grid) & (grid < deaths)).sum(axis=0).astype(float)

def gaussian_smooth(curve: np.ndarray, sigma: float = BETTI_ARC_SMOOTH_SIGMA) -> np.ndarray:
    if sigma <= 0:
        return curve
    radius = max(1, int(np.ceil(3 * sigma)))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum()
    padded = np.pad(curve, pad_width=radius, mode='edge')
    return np.convolve(padded, kernel, mode='valid')

def get_real_anchor_table() -> pd.DataFrame:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    keep_cols = [
        'session_id',
        'switch_index',
        'switch_time_idx',
        'transition_type',
        'posterior_jump_l2',
        'segment_total_persistence_h1',
        'segment_betti_h1_auc',
    ]
    return (
        switch_df[keep_cols]
        .drop_duplicates(['session_id', 'switch_index'])
        .sort_values(['session_id', 'switch_index'])
        .reset_index(drop=True)
    )

def anchor_has_full_support(session_id: str, switch_time_idx: int) -> bool:
    n = len(get_cached_posterior(session_id))
    return (switch_time_idx - 30 >= 0) and (switch_time_idx + 30 < n)

def sample_real_anchor_table(sample_size: int = BETTI_ARC_SAMPLE_SIZE, seed: int = BETTI_ARC_RANDOM_SEED) -> pd.DataFrame:
    anchor_table = get_real_anchor_table()
    support_mask = [anchor_has_full_support(str(row.session_id), int(row.switch_time_idx)) for row in anchor_table.itertuples(index=False)]
    anchor_table = anchor_table.loc[support_mask].copy()
    if len(anchor_table) > sample_size:
        anchor_table = anchor_table.sample(sample_size, random_state=seed)
    return anchor_table.sort_values(['session_id', 'switch_index']).reset_index(drop=True)

def select_transition_dims(session_id: str, switch_time_idx: int, n_dims: int = 3) -> list[int]:
    posterior = get_cached_posterior(session_id)
    start = max(0, switch_time_idx - 30)
    end = min(len(posterior), switch_time_idx + 31)
    local = posterior[start:end]
    variances = local.var(axis=0)
    order = np.argsort(variances)[::-1]
    return order[:n_dims].tolist()

def h1_diagram_for_real_window(session_id: str, switch_time_idx: int, center_sec: int, dims: list[int], half_width_sec: int = BETTI_ARC_HALF_WIDTH) -> np.ndarray:
    posterior = get_cached_posterior(session_id)[:, dims]
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)
    start = switch_time_idx + center_sec - half_width_sec
    end = switch_time_idx + center_sec + half_width_sec
    mask = (centers >= start) & (centers <= end)
    points = embedded[mask]
    if len(points) < 5:
        return np.empty((0, 2), dtype=float)
    diagram = np.asarray(ripser(points, maxdim=1)['dgms'][1], dtype=float)
    if diagram.size == 0:
        return np.empty((0, 2), dtype=float)
    finite = np.isfinite(diagram).all(axis=1)
    return diagram[finite]

anchor_table = sample_real_anchor_table()
diagram_rows = []
max_death = 0.0

for _, row in anchor_table.iterrows():
    dims = select_transition_dims(str(row['session_id']), int(row['switch_time_idx']), n_dims=3)
    for center_sec in BETTI_ARC_CENTERS:
        diagram = h1_diagram_for_real_window(
            session_id=str(row['session_id']),
            switch_time_idx=int(row['switch_time_idx']),
            center_sec=int(center_sec),
            dims=dims,
        )
        if len(diagram):
            max_death = max(max_death, float(diagram[:, 1].max()))
        diagram_rows.append({
            'session_id': str(row['session_id']),
            'switch_index': int(row['switch_index']),
            'switch_time_idx': int(row['switch_time_idx']),
            'center_sec': int(center_sec),
            'diagram': diagram,
        })

if max_death <= 0:
    raise RuntimeError('No finite H1 deaths found for the sampled transition windows.')

grid = np.linspace(0.0, max_death, BETTI_ARC_GRID_SIZE)
curve_rows = []
summary_rows = []
center_norm = Normalize(vmin=min(BETTI_ARC_CENTERS), vmax=max(BETTI_ARC_CENTERS))
cmap = plt.get_cmap('coolwarm')

for center_sec in BETTI_ARC_CENTERS:
    center_diagrams = [row['diagram'] for row in diagram_rows if row['center_sec'] == center_sec]
    curves = np.stack([gaussian_smooth(betti_curve_from_diagram(diagram, grid)) for diagram in center_diagrams], axis=0)
    mean_curve = curves.mean(axis=0)
    q25_curve = np.quantile(curves, 0.25, axis=0)
    q75_curve = np.quantile(curves, 0.75, axis=0)
    peak_idx = int(np.argmax(mean_curve))
    summary_rows.append({
        'center_sec': int(center_sec),
        'n_transitions': int(curves.shape[0]),
        'peak_beta_1': float(mean_curve[peak_idx]),
        'peak_filtration_value': float(grid[peak_idx]),
        'mean_auc': float(np.trapezoid(mean_curve, grid)),
        'mean_beta_1_at_midscale': float(np.interp(0.5 * max_death, grid, mean_curve)),
    })
    for filtration_value, mean_beta_1, q25_beta_1, q75_beta_1 in zip(grid, mean_curve, q25_curve, q75_curve, strict=False):
        curve_rows.append({
            'center_sec': int(center_sec),
            'filtration_value': float(filtration_value),
            'mean_beta_1': float(mean_beta_1),
            'q25_beta_1': float(q25_beta_1),
            'q75_beta_1': float(q75_beta_1),
        })

df_betti_arc = pd.DataFrame(curve_rows)
df_betti_arc_summary = pd.DataFrame(summary_rows).sort_values('center_sec').reset_index(drop=True)
df_betti_arc.to_csv(BETTI_ARC_CURVES_PATH, index=False)
df_betti_arc_summary.to_csv(BETTI_ARC_SUMMARY_PATH, index=False)

fig, ax = plt.subplots(figsize=(9.2, 5.4), constrained_layout=True)

for center_sec in BETTI_ARC_CENTERS:
    color = cmap(center_norm(center_sec))
    sub = df_betti_arc[df_betti_arc['center_sec'] == center_sec].copy()
    ax.fill_between(
        sub['filtration_value'],
        sub['q25_beta_1'],
        sub['q75_beta_1'],
        color=color,
        alpha=0.12,
        linewidth=0,
    )
    ax.plot(
        sub['filtration_value'],
        sub['mean_beta_1'],
        color=color,
        linewidth=2.2,
        label=f'{center_sec:+d}s',
    )

ax.set_title('Mean Betti Curves Across the Transition Arc', fontsize=14)
ax.set_xlabel('Filtration value', fontsize=11)
ax.set_ylabel('Mean beta_1(epsilon)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
sm = plt.cm.ScalarMappable(norm=center_norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Window center (s)', fontsize=10)

summary_text = (
    f'{len(anchor_table)} real transitions with full ±30s support | '
    f'window half-width={BETTI_ARC_HALF_WIDTH}s | '
    f'delay embedding m={DELAY_M}, tau={DELAY_TAU}'
)
fig.suptitle(summary_text, fontsize=10, y=1.02)
fig.savefig(BETTI_ARC_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(BETTI_ARC_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

display(df_betti_arc_summary)
print('Saved SVG to', BETTI_ARC_FIG_PATH)
print('Saved PNG to', BETTI_ARC_PNG_PATH)


In [ ]:
from ripser import ripser

BETTI_TIME_FIG_PATH = FIG_DIR / 'topo_transition_real_betti_auc_timecourse.svg'
BETTI_TIME_PNG_PATH = FIG_DIR / 'topo_transition_real_betti_auc_timecourse.png'
BETTI_TIME_CURVES_PATH = TAB_DIR / 'topo_transition_real_betti_auc_timecourse.csv'
BETTI_TIME_SUMMARY_PATH = TAB_DIR / 'topo_transition_real_betti_auc_timecourse_summary.csv'

BETTI_TIME_OFFSETS = np.arange(-30, 31, 1)
BETTI_TIME_HALF_WIDTH = 5
BETTI_TIME_SAMPLE_SIZE = 300
BETTI_TIME_RANDOM_SEED = 13
BETTI_TIME_GRID_SIZE = 180
BETTI_TIME_CURVE_SMOOTH_SIGMA = 1.6
BETTI_TIME_SERIES_SMOOTH_SIGMA = 1.5
BETTI_TIME_N_DIMS = 3

POSTERIOR_CACHE_TIME: dict[str, np.ndarray] = {}

def get_cached_posterior_time(session_id: str) -> np.ndarray:
    if session_id not in POSTERIOR_CACHE_TIME:
        POSTERIOR_CACHE_TIME[session_id] = posterior_matrix(MODEL, SESSION_PATHS[session_id])
    return POSTERIOR_CACHE_TIME[session_id]

def gaussian_smooth_1d(curve: np.ndarray, sigma: float) -> np.ndarray:
    if sigma <= 0:
        return curve.astype(float, copy=True)
    radius = max(1, int(np.ceil(3 * sigma)))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum()
    padded = np.pad(curve, pad_width=radius, mode='edge')
    return np.convolve(padded, kernel, mode='valid')

def betti_curve_from_diagram_time(diagram: np.ndarray, grid: np.ndarray) -> np.ndarray:
    if diagram.size == 0:
        return np.zeros_like(grid)
    births = diagram[:, 0][:, None]
    deaths = diagram[:, 1][:, None]
    return ((births <= grid) & (grid < deaths)).sum(axis=0).astype(float)

def get_real_anchor_table_time() -> pd.DataFrame:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    keep_cols = [
        'session_id',
        'switch_index',
        'switch_time_idx',
        'transition_type',
        'posterior_jump_l2',
        'segment_total_persistence_h1',
    ]
    return (
        switch_df[keep_cols]
        .drop_duplicates(['session_id', 'switch_index'])
        .sort_values(['session_id', 'switch_index'])
        .reset_index(drop=True)
    )

def anchor_has_full_support_time(session_id: str, switch_time_idx: int) -> bool:
    n = len(get_cached_posterior_time(session_id))
    pad = int(np.max(np.abs(BETTI_TIME_OFFSETS)) + BETTI_TIME_HALF_WIDTH)
    return (switch_time_idx - pad >= 0) and (switch_time_idx + pad < n)

def sample_real_anchor_table_time(sample_size: int = BETTI_TIME_SAMPLE_SIZE, seed: int = BETTI_TIME_RANDOM_SEED) -> pd.DataFrame:
    anchor_table = get_real_anchor_table_time()
    support_mask = [anchor_has_full_support_time(str(row.session_id), int(row.switch_time_idx)) for row in anchor_table.itertuples(index=False)]
    anchor_table = anchor_table.loc[support_mask].copy()
    if len(anchor_table) > sample_size:
        anchor_table = anchor_table.sample(sample_size, random_state=seed)
    return anchor_table.sort_values(['session_id', 'switch_index']).reset_index(drop=True)

def select_transition_dims_time(session_id: str, switch_time_idx: int, n_dims: int = BETTI_TIME_N_DIMS) -> list[int]:
    posterior = get_cached_posterior_time(session_id)
    pad = int(np.max(np.abs(BETTI_TIME_OFFSETS)) + BETTI_TIME_HALF_WIDTH)
    local = posterior[switch_time_idx - pad:switch_time_idx + pad + 1]
    variances = local.var(axis=0)
    order = np.argsort(variances)[::-1]
    return order[:n_dims].tolist()

def h1_diagram_for_offset(session_id: str, switch_time_idx: int, offset_sec: int, dims: list[int]) -> np.ndarray:
    posterior = get_cached_posterior_time(session_id)[:, dims]
    embedded, centers = build_delay_embedding(posterior, m=DELAY_M, tau=DELAY_TAU)
    start = switch_time_idx + offset_sec - BETTI_TIME_HALF_WIDTH
    end = switch_time_idx + offset_sec + BETTI_TIME_HALF_WIDTH
    mask = (centers >= start) & (centers <= end)
    points = embedded[mask]
    if len(points) < 5:
        return np.empty((0, 2), dtype=float)
    diagram = np.asarray(ripser(points, maxdim=1)['dgms'][1], dtype=float)
    if diagram.size == 0:
        return np.empty((0, 2), dtype=float)
    finite = np.isfinite(diagram).all(axis=1)
    return diagram[finite]

anchor_table = sample_real_anchor_table_time()
diagram_payloads = []
max_death = 0.0

for _, row in anchor_table.iterrows():
    session_id = str(row['session_id'])
    switch_time_idx = int(row['switch_time_idx'])
    dims = select_transition_dims_time(session_id, switch_time_idx)
    for offset_sec in BETTI_TIME_OFFSETS:
        diagram = h1_diagram_for_offset(session_id, switch_time_idx, int(offset_sec), dims)
        if len(diagram):
            max_death = max(max_death, float(diagram[:, 1].max()))
        diagram_payloads.append({
            'session_id': session_id,
            'switch_index': int(row['switch_index']),
            'switch_time_idx': switch_time_idx,
            'offset_sec': int(offset_sec),
            'diagram': diagram,
        })

if max_death <= 0:
    raise RuntimeError('No finite H1 deaths found for the sampled time-course windows.')

grid = np.linspace(0.0, max_death, BETTI_TIME_GRID_SIZE)
rows = []
summary_rows = []

for offset_sec in BETTI_TIME_OFFSETS:
    diagrams = [row['diagram'] for row in diagram_payloads if row['offset_sec'] == offset_sec]
    curves = np.stack([
        gaussian_smooth_1d(betti_curve_from_diagram_time(diagram, grid), BETTI_TIME_CURVE_SMOOTH_SIGMA)
        for diagram in diagrams
    ], axis=0)
    aucs = np.trapezoid(curves, grid, axis=1)
    rows.extend([
        {
            'offset_sec': int(offset_sec),
            'mean_auc': float(aucs.mean()),
            'q25_auc': float(np.quantile(aucs, 0.25)),
            'q75_auc': float(np.quantile(aucs, 0.75)),
            'sem_auc': float(aucs.std(ddof=1) / np.sqrt(len(aucs))),
        }
    ])
    summary_rows.append({
        'offset_sec': int(offset_sec),
        'n_transitions': int(len(aucs)),
        'mean_auc_raw': float(aucs.mean()),
        'median_auc_raw': float(np.median(aucs)),
    })

df_timecourse = pd.DataFrame(rows).sort_values('offset_sec').reset_index(drop=True)
df_timecourse['mean_auc_smooth'] = gaussian_smooth_1d(df_timecourse['mean_auc'].to_numpy(), BETTI_TIME_SERIES_SMOOTH_SIGMA)
df_timecourse['q25_auc_smooth'] = gaussian_smooth_1d(df_timecourse['q25_auc'].to_numpy(), BETTI_TIME_SERIES_SMOOTH_SIGMA)
df_timecourse['q75_auc_smooth'] = gaussian_smooth_1d(df_timecourse['q75_auc'].to_numpy(), BETTI_TIME_SERIES_SMOOTH_SIGMA)
df_timecourse['sem_auc_smooth'] = gaussian_smooth_1d(df_timecourse['sem_auc'].to_numpy(), BETTI_TIME_SERIES_SMOOTH_SIGMA)
df_timecourse.to_csv(BETTI_TIME_CURVES_PATH, index=False)

df_time_summary = pd.DataFrame(summary_rows).sort_values('offset_sec').reset_index(drop=True)
peak_idx = int(df_timecourse['mean_auc_smooth'].to_numpy().argmax())
peak_row = df_timecourse.iloc[peak_idx]
summary_table = pd.DataFrame([
    {
        'n_transitions': int(len(anchor_table)),
        'offset_with_peak_auc_sec': int(peak_row['offset_sec']),
        'peak_mean_auc_smooth': float(peak_row['mean_auc_smooth']),
        'pre_mean_auc_smooth': float(df_timecourse.loc[df_timecourse['offset_sec'] == -5, 'mean_auc_smooth'].iloc[0]),
        'post_mean_auc_smooth': float(df_timecourse.loc[df_timecourse['offset_sec'] == 5, 'mean_auc_smooth'].iloc[0]),
        'late_mean_auc_smooth': float(df_timecourse.loc[df_timecourse['offset_sec'] == 25, 'mean_auc_smooth'].iloc[0]),
    }
])
summary_table.to_csv(BETTI_TIME_SUMMARY_PATH, index=False)

fig, ax = plt.subplots(figsize=(8.8, 4.9), constrained_layout=True)
ax.fill_between(
    df_timecourse['offset_sec'],
    df_timecourse['q25_auc_smooth'],
    df_timecourse['q75_auc_smooth'],
    color='#7aa6ff',
    alpha=0.22,
    linewidth=0,
)
ax.plot(df_timecourse['offset_sec'], df_timecourse['mean_auc_smooth'], color='#1f4fbf', linewidth=2.8)
ax.axvline(0, color='0.45', linestyle='--', linewidth=1.2)
ax.set_title('Topological Complexity Time-Course Around Real Transitions', fontsize=14)
ax.set_xlabel('Time relative to switch (s)', fontsize=11)
ax.set_ylabel('Betti-curve AUC', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
subtitle = (
    f'{len(anchor_table)} real transitions with full ±35s support | '
    f'11s local windows | top-{BETTI_TIME_N_DIMS} posterior dimensions per transition'
)
fig.suptitle(subtitle, fontsize=10, y=1.02)
fig.savefig(BETTI_TIME_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(BETTI_TIME_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

display(summary_table)
print('Saved SVG to', BETTI_TIME_FIG_PATH)
print('Saved PNG to', BETTI_TIME_PNG_PATH)


In [ ]:
from ripser import ripser
import pyarrow.parquet as pq
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch
from sklearn.decomposition import PCA

PED_FIG_PATH = FIG_DIR / 'topo_transition_pedagogical_panel.svg'
PED_PNG_PATH = FIG_DIR / 'topo_transition_pedagogical_panel.png'
PED_SUMMARY_PATH = TAB_DIR / 'topo_transition_pedagogical_panel_summary.csv'
PED_POINTS_PATH = TAB_DIR / 'topo_transition_pedagogical_panel_points.csv'

PED_EXAMPLE = {
    'session_id': '154_A',
    'switch_index': 406,
    'switch_time_idx': 1278,
}
PED_WINDOWS = [
    ('Pre', -30, -5),
    ('Peri', -5, 5),
    ('Post', 5, 20),
]

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
plt.rcParams['svg.fonttype'] = 'path'

def load_affect_window(session_id: str, switch_time_idx: int, half_width: int = 30) -> pd.DataFrame:
    table = pq.read_table(SESSION_PATHS[session_id])
    df = table.to_pandas()
    sub = df.iloc[switch_time_idx - half_width:switch_time_idx + half_width + 1].copy().reset_index(drop=True)
    sub['rel_time'] = np.arange(-half_width, half_width + 1)
    return sub

def select_local_dims(posterior: np.ndarray, switch_time_idx: int, n_dims: int = 3) -> list[int]:
    local = posterior[switch_time_idx - 30:switch_time_idx + 31]
    order = np.argsort(local.var(axis=0))[::-1]
    return order[:n_dims].tolist()

def longest_h1(diagram: np.ndarray) -> np.ndarray | None:
    if diagram.size == 0:
        return None
    life = diagram[:, 1] - diagram[:, 0]
    if len(life) == 0:
        return None
    return diagram[int(np.argmax(life))]

example = PED_EXAMPLE.copy()
session_id = example['session_id']
switch_time_idx = int(example['switch_time_idx'])
affect_df = load_affect_window(session_id, switch_time_idx)
posterior = posterior_matrix(MODEL, SESSION_PATHS[session_id])
dims = select_local_dims(posterior, switch_time_idx, n_dims=3)
embedded, centers = build_delay_embedding(posterior[:, dims], m=DELAY_M, tau=DELAY_TAU)
plot_mask = (centers >= switch_time_idx - 30) & (centers <= switch_time_idx + 20)
plot_points = embedded[plot_mask]
plot_centers = centers[plot_mask]
proj = PCA(n_components=2).fit_transform(plot_points)
proj_df = pd.DataFrame({
    'time_idx': plot_centers,
    'rel_time': plot_centers - switch_time_idx,
    'pc1': proj[:, 0],
    'pc2': proj[:, 1],
})

diagram_rows = []
summary_rows = []
for window_name, start_sec, end_sec in PED_WINDOWS:
    mask = (proj_df['rel_time'] >= start_sec) & (proj_df['rel_time'] <= end_sec)
    pts = proj_df.loc[mask, ['pc1', 'pc2']].to_numpy()
    path = float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum()) if len(pts) > 1 else 0.0
    disp = float(np.linalg.norm(pts[-1] - pts[0])) if len(pts) > 1 else 0.0
    tort = path / (disp + 1e-9)
    emb_mask = (centers >= switch_time_idx + start_sec) & (centers <= switch_time_idx + end_sec)
    diag = np.asarray(ripser(embedded[emb_mask], maxdim=1)['dgms'][1], dtype=float)
    diag = diag[np.isfinite(diag).all(axis=1)] if diag.size else np.empty((0, 2), dtype=float)
    longest = longest_h1(diag)
    summary_rows.append({
        'window': window_name,
        'n_points': int(mask.sum()),
        'path_length': path,
        'displacement': disp,
        'tortuosity': tort,
        'h1_birth': float(longest[0]) if longest is not None else np.nan,
        'h1_death': float(longest[1]) if longest is not None else np.nan,
        'h1_lifetime': float(longest[1] - longest[0]) if longest is not None else np.nan,
    })
    for birth, death in diag:
        diagram_rows.append({'window': window_name, 'birth': float(birth), 'death': float(death), 'lifetime': float(death - birth)})

df_summary = pd.DataFrame(summary_rows)
df_summary.insert(0, 'session_id', session_id)
df_summary.insert(1, 'switch_index', int(example['switch_index']))
df_summary.insert(2, 'switch_time_idx', switch_time_idx)
df_summary.insert(3, 'projection_dims', ','.join(map(str, dims)))
df_summary.to_csv(PED_SUMMARY_PATH, index=False)
proj_df.to_csv(PED_POINTS_PATH, index=False)

raster_values = np.vstack([affect_df['T'].to_numpy(), affect_df['C'].to_numpy()])
raster_cmap = ListedColormap(['#8C1D18', '#7A7A7A', '#2A9D8F'])
raster_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], raster_cmap.N)
time_norm = Normalize(vmin=-30, vmax=20)
time_cmap = plt.get_cmap('viridis')

xpad = 0.08 * max(proj_df['pc1'].max() - proj_df['pc1'].min(), 1e-6)
ypad = 0.08 * max(proj_df['pc2'].max() - proj_df['pc2'].min(), 1e-6)
xlim = (proj_df['pc1'].min() - xpad, proj_df['pc1'].max() + xpad)
ylim = (proj_df['pc2'].min() - ypad, proj_df['pc2'].max() + ypad)
peri_diag = pd.DataFrame(diagram_rows)
peri_diag = peri_diag[peri_diag['window'] == 'Peri'].copy()
diag_max = 1.0
if not peri_diag.empty:
    diag_max = max(diag_max, float(peri_diag[['birth', 'death']].to_numpy().max()) * 1.08)
peri_longest = None
if not peri_diag.empty:
    peri_longest = peri_diag.iloc[int(np.argmax(peri_diag['lifetime'].to_numpy()))]

fig = plt.figure(figsize=(13.6, 9.4), constrained_layout=False)
gs = GridSpec(4, 5, figure=fig, height_ratios=[0.9, 1.2, 1.25, 0.62], width_ratios=[1.0, 1.0, 1.0, 0.08, 1.15], hspace=0.24, wspace=0.32)

ax_raster = fig.add_subplot(gs[0, :])
ax_post = fig.add_subplot(gs[1, :])
traj_axes = [fig.add_subplot(gs[2, i]) for i in range(3)]
ax_traj_cbar = fig.add_subplot(gs[2, 3])
ax_pd = fig.add_subplot(gs[2, 4])
ax_caption = fig.add_subplot(gs[3, :])

im = ax_raster.imshow(raster_values, aspect='auto', cmap=raster_cmap, norm=raster_norm, extent=[-30.5, 30.5, 1.5, -0.5])
ax_raster.set_yticks([0, 1])
ax_raster.set_yticklabels(['Therapist', 'Client'], fontsize=10)
ax_raster.set_xticks([])
ax_raster.set_title('1. Raw Affect Codes', loc='left', fontsize=13, fontweight='bold')
ax_raster.axvline(0, color='black', linestyle='--', linewidth=1.2)
for spine in ['top', 'right', 'bottom']:
    ax_raster.spines[spine].set_visible(False)
legend_handles = [
    plt.Line2D([0], [0], color='#8C1D18', lw=8),
    plt.Line2D([0], [0], color='#7A7A7A', lw=8),
    plt.Line2D([0], [0], color='#2A9D8F', lw=8),
]
ax_raster.legend(legend_handles, ['Negative', 'Neutral', 'Positive'], frameon=False, ncol=3, bbox_to_anchor=(1.0, 1.35), loc='upper right', fontsize=9)

posterior_window = posterior[switch_time_idx - 30:switch_time_idx + 31].T
post_im = ax_post.imshow(posterior_window, aspect='auto', cmap='magma', vmin=0, vmax=1, extent=[-30.5, 30.5, 7.5, -0.5])
ax_post.axvline(0, color='white', linestyle='--', linewidth=1.2)
ax_post.set_yticks(range(8))
ax_post.set_yticklabels([f'State {k}' for k in range(8)], fontsize=9)
ax_post.set_xticks([])
ax_post.set_title('2. Regime Posterior Probabilities', loc='left', fontsize=13, fontweight='bold')
for spine in ['top', 'right', 'bottom']:
    ax_post.spines[spine].set_visible(False)
cbar_post = fig.colorbar(post_im, ax=ax_post, fraction=0.015, pad=0.01)
cbar_post.set_label('Posterior mass', fontsize=9)

for ax, (window_name, start_sec, end_sec) in zip(traj_axes, PED_WINDOWS, strict=False):
    sub = proj_df[(proj_df['rel_time'] >= start_sec) & (proj_df['rel_time'] <= end_sec)].copy()
    ax.plot(sub['pc1'], sub['pc2'], color='0.75', linewidth=1.1, zorder=1)
    sc = ax.scatter(sub['pc1'], sub['pc2'], c=sub['rel_time'], cmap=time_cmap, norm=time_norm, s=34, edgecolor='white', linewidth=0.5, zorder=2)
    ax.scatter(sub['pc1'].iloc[0], sub['pc2'].iloc[0], s=62, facecolor='white', edgecolor='black', linewidth=1.0, zorder=3)
    ax.scatter(sub['pc1'].iloc[-1], sub['pc2'].iloc[-1], s=62, facecolor='black', edgecolor='black', linewidth=0.8, zorder=3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'{window_name} ({start_sec:+d}s to {end_sec:+d}s)', fontsize=11)
    ax.set_xlabel('PC1', fontsize=10)
    if ax is traj_axes[0]:
        ax.set_ylabel('PC2', fontsize=10)
    else:
        ax.set_yticklabels([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.text(0.08, 0.46, '3. Delay-Embedded Posterior Geometry', fontsize=13, fontweight='bold', ha='left')
cbar_traj = fig.colorbar(plt.cm.ScalarMappable(norm=time_norm, cmap=time_cmap), cax=ax_traj_cbar)
cbar_traj.set_label('Time relative to switch (s)', fontsize=9)

if not peri_diag.empty:
    ax_pd.scatter(peri_diag['birth'], peri_diag['death'], color='#1f77b4', s=42, alpha=0.85)
ax_pd.plot([0, diag_max], [0, diag_max], linestyle='--', color='0.55', linewidth=1.0)
ax_pd.set_xlim(0, diag_max)
ax_pd.set_ylim(0, diag_max)
ax_pd.set_aspect('equal', adjustable='box')
ax_pd.set_xlabel('Birth', fontsize=10)
ax_pd.set_ylabel('Death', fontsize=10)
ax_pd.set_title('4. Peri-Switch Persistence Diagram', loc='left', fontsize=13, fontweight='bold', pad=10)
ax_pd.spines['top'].set_visible(False)
ax_pd.spines['right'].set_visible(False)
if peri_longest is not None:
    ax_pd.scatter([peri_longest['birth']], [peri_longest['death']], color='#D55E00', s=68, zorder=3)
    ax_pd.annotate(
        'Longest-lived H1 loop',
        xy=(peri_longest['birth'], peri_longest['death']),
        xytext=(peri_longest['birth'] + 0.1, peri_longest['death'] + 0.12),
        arrowprops=dict(arrowstyle='->', lw=1.0, color='#D55E00'),
        fontsize=9,
        color='#7A3B00',
    )

ax_caption.axis('off')
ax_caption.set_xlim(0, 1)
ax_caption.set_ylim(0, 1)
peri_summary = df_summary[df_summary['window'] == 'Peri'].iloc[0]
caption = (
    'Same moment, four views: observable affect codes, regime posteriors, embedded geometry, and topology. '
    f'For session {session_id}, switch {int(example['switch_index'])} at t={switch_time_idx}s, posterior mass rotates across states 1, 7, and 3 near the switch, '
    'the peri-switch trajectory forms the clearest loop, and the persistence diagram records that loop as a single off-diagonal H1 generator. '
    f'Peri-switch H1 lifetime = {peri_summary['h1_lifetime']:.2f}.'
)
ax_caption.text(0.0, 0.92, caption, va='top', ha='left', fontsize=10.5, wrap=True)

fig.suptitle('From Affect Codes to Topology Across a Single Regime Transition', fontsize=17, y=0.985)
fig.subplots_adjust(top=0.86, left=0.08, right=0.96, bottom=0.08)
fig.savefig(PED_FIG_PATH, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(PED_PNG_PATH, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Saved SVG to', PED_FIG_PATH)
print('Saved PNG to', PED_PNG_PATH)
display(df_summary)


In [ ]:
from ripser import ripser
import pyarrow.parquet as pq
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.decomposition import PCA

PED_SEP_PREFIX = 'topo_transition_pedagogical'
PED_RASTER_SVG = FIG_DIR / f'{PED_SEP_PREFIX}_raster.svg'
PED_RASTER_PNG = FIG_DIR / f'{PED_SEP_PREFIX}_raster.png'
PED_POST_SVG = FIG_DIR / f'{PED_SEP_PREFIX}_posterior.svg'
PED_POST_PNG = FIG_DIR / f'{PED_SEP_PREFIX}_posterior.png'
PED_TRAJ_SVG = FIG_DIR / f'{PED_SEP_PREFIX}_trajectory.svg'
PED_TRAJ_PNG = FIG_DIR / f'{PED_SEP_PREFIX}_trajectory.png'
PED_PD_SVG = FIG_DIR / f'{PED_SEP_PREFIX}_persistence.svg'
PED_PD_PNG = FIG_DIR / f'{PED_SEP_PREFIX}_persistence.png'

PED_EXAMPLE = {
    'session_id': '154_A',
    'switch_index': 406,
    'switch_time_idx': 1278,
}
PED_WINDOWS = [
    ('Pre', -30, -5),
    ('Peri', -5, 5),
    ('Post', 5, 20),
]

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
plt.rcParams['svg.fonttype'] = 'path'

def load_affect_window_clean(session_id: str, switch_time_idx: int, half_width: int = 30) -> pd.DataFrame:
    table = pq.read_table(SESSION_PATHS[session_id])
    df = table.to_pandas()
    sub = df.iloc[switch_time_idx - half_width:switch_time_idx + half_width + 1].copy().reset_index(drop=True)
    sub['rel_time'] = np.arange(-half_width, half_width + 1)
    return sub

def select_local_dims_clean(posterior: np.ndarray, switch_time_idx: int, n_dims: int = 3) -> list[int]:
    local = posterior[switch_time_idx - 30:switch_time_idx + 31]
    order = np.argsort(local.var(axis=0))[::-1]
    return order[:n_dims].tolist()

def longest_h1_clean(diagram: np.ndarray) -> np.ndarray | None:
    if diagram.size == 0:
        return None
    life = diagram[:, 1] - diagram[:, 0]
    if len(life) == 0:
        return None
    return diagram[int(np.argmax(life))]

example = PED_EXAMPLE.copy()
session_id = example['session_id']
switch_time_idx = int(example['switch_time_idx'])
affect_df = load_affect_window_clean(session_id, switch_time_idx)
posterior = posterior_matrix(MODEL, SESSION_PATHS[session_id])
dims = select_local_dims_clean(posterior, switch_time_idx, n_dims=3)
embedded, centers = build_delay_embedding(posterior[:, dims], m=DELAY_M, tau=DELAY_TAU)
plot_mask = (centers >= switch_time_idx - 30) & (centers <= switch_time_idx + 20)
plot_points = embedded[plot_mask]
plot_centers = centers[plot_mask]
proj = PCA(n_components=2).fit_transform(plot_points)
proj_df = pd.DataFrame({
    'time_idx': plot_centers,
    'rel_time': plot_centers - switch_time_idx,
    'pc1': proj[:, 0],
    'pc2': proj[:, 1],
})

diagram_rows = []
for window_name, start_sec, end_sec in PED_WINDOWS:
    emb_mask = (centers >= switch_time_idx + start_sec) & (centers <= switch_time_idx + end_sec)
    diag = np.asarray(ripser(embedded[emb_mask], maxdim=1)['dgms'][1], dtype=float)
    diag = diag[np.isfinite(diag).all(axis=1)] if diag.size else np.empty((0, 2), dtype=float)
    for birth, death in diag:
        diagram_rows.append({'window': window_name, 'birth': float(birth), 'death': float(death), 'lifetime': float(death - birth)})
df_diag = pd.DataFrame(diagram_rows)
peri_diag = df_diag[df_diag['window'] == 'Peri'].copy()
peri_longest = None if peri_diag.empty else peri_diag.iloc[int(np.argmax(peri_diag['lifetime'].to_numpy()))]

raster_values = np.vstack([affect_df['T'].to_numpy(), affect_df['C'].to_numpy()])
raster_cmap = ListedColormap(['#8C1D18', '#7A7A7A', '#2A9D8F'])
raster_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], raster_cmap.N)
time_norm = Normalize(vmin=-30, vmax=20)
time_cmap = plt.get_cmap('viridis')
xpad = 0.08 * max(proj_df['pc1'].max() - proj_df['pc1'].min(), 1e-6)
ypad = 0.08 * max(proj_df['pc2'].max() - proj_df['pc2'].min(), 1e-6)
xlim = (proj_df['pc1'].min() - xpad, proj_df['pc1'].max() + xpad)
ylim = (proj_df['pc2'].min() - ypad, proj_df['pc2'].max() + ypad)
diag_max = 1.0 if peri_diag.empty else max(1.0, float(peri_diag[['birth', 'death']].to_numpy().max()) * 1.08)

# Panel 1: raster
fig, ax = plt.subplots(figsize=(12.2, 1.9))
ax.imshow(raster_values, aspect='auto', cmap=raster_cmap, norm=raster_norm, extent=[-30.5, 30.5, 1.5, -0.5])
ax.set_yticks([0, 1])
ax.set_yticklabels(['Therapist', 'Client'], fontsize=10)
ax.set_xticks([-30, -20, -10, 0, 10, 20, 30])
ax.set_xlabel('Time relative to switch (s)', fontsize=10)
ax.axvline(0, color='black', linestyle='--', linewidth=1.1)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
fig.savefig(PED_RASTER_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(PED_RASTER_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 2: posterior heatmap
fig, ax = plt.subplots(figsize=(12.2, 2.3))
posterior_window = posterior[switch_time_idx - 30:switch_time_idx + 31].T
im = ax.imshow(posterior_window, aspect='auto', cmap='magma', vmin=0, vmax=1, extent=[-30.5, 30.5, 7.5, -0.5])
ax.axvline(0, color='white', linestyle='--', linewidth=1.1)
ax.set_yticks(range(8))
ax.set_yticklabels([f'State {k}' for k in range(8)], fontsize=9)
ax.set_xticks([-30, -20, -10, 0, 10, 20, 30])
ax.set_xlabel('Time relative to switch (s)', fontsize=10)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.018, pad=0.01)
cbar.set_label('Posterior mass', fontsize=9)
fig.savefig(PED_POST_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(PED_POST_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 3: trajectories
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.25), sharex=True, sharey=True)
for ax, (window_name, start_sec, end_sec) in zip(axes, PED_WINDOWS, strict=False):
    sub = proj_df[(proj_df['rel_time'] >= start_sec) & (proj_df['rel_time'] <= end_sec)].copy()
    ax.plot(sub['pc1'], sub['pc2'], color='0.78', linewidth=1.1, zorder=1)
    ax.scatter(sub['pc1'], sub['pc2'], c=sub['rel_time'], cmap=time_cmap, norm=time_norm, s=34, edgecolor='white', linewidth=0.45, zorder=2)
    ax.scatter(sub['pc1'].iloc[0], sub['pc2'].iloc[0], s=60, facecolor='white', edgecolor='black', linewidth=1.0, zorder=3)
    ax.scatter(sub['pc1'].iloc[-1], sub['pc2'].iloc[-1], s=60, facecolor='black', edgecolor='black', linewidth=0.8, zorder=3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(window_name, fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel('PC2', fontsize=10)
for ax in axes:
    ax.set_xlabel('PC1', fontsize=10)
sm = plt.cm.ScalarMappable(norm=time_norm, cmap=time_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.02)
cbar.set_label('Time relative to switch (s)', fontsize=9)
fig.savefig(PED_TRAJ_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(PED_TRAJ_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 4: persistence diagram
fig, ax = plt.subplots(figsize=(4.0, 3.5))
if not peri_diag.empty:
    ax.scatter(peri_diag['birth'], peri_diag['death'], color='#1f77b4', s=46, alpha=0.9)
if peri_longest is not None:
    ax.scatter([peri_longest['birth']], [peri_longest['death']], color='#D55E00', s=72, zorder=3)
ax.plot([0, diag_max], [0, diag_max], linestyle='--', color='0.6', linewidth=1.0)
ax.set_xlim(0, diag_max)
ax.set_ylim(0, diag_max)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Birth', fontsize=10)
ax.set_ylabel('Death', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.savefig(PED_PD_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(PED_PD_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print('Saved:', PED_RASTER_SVG)
print('Saved:', PED_POST_SVG)
print('Saved:', PED_TRAJ_SVG)
print('Saved:', PED_PD_SVG)


In [ ]:
from ripser import ripser
import pyarrow.parquet as pq
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.decomposition import PCA

TOP5_PREFIX = 'topo_transition_top5_clean'
TOP5_RASTER_SVG = FIG_DIR / f'{TOP5_PREFIX}_raster.svg'
TOP5_RASTER_PNG = FIG_DIR / f'{TOP5_PREFIX}_raster.png'
TOP5_POST_SVG = FIG_DIR / f'{TOP5_PREFIX}_posterior.svg'
TOP5_POST_PNG = FIG_DIR / f'{TOP5_PREFIX}_posterior.png'
TOP5_TRAJ_SVG = FIG_DIR / f'{TOP5_PREFIX}_trajectory.svg'
TOP5_TRAJ_PNG = FIG_DIR / f'{TOP5_PREFIX}_trajectory.png'
TOP5_PD_SVG = FIG_DIR / f'{TOP5_PREFIX}_persistence.svg'
TOP5_PD_PNG = FIG_DIR / f'{TOP5_PREFIX}_persistence.png'
TOP5_SUMMARY_PATH = TAB_DIR / f'{TOP5_PREFIX}_summary.csv'

GEOM_WINDOWS = [('Pre', -30, -8), ('Peri', -8, 8), ('Post', 8, 25)]

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
plt.rcParams['svg.fonttype'] = 'path'

def load_session_window(session_id: str, switch_time_idx: int, half_width: int = 30) -> pd.DataFrame:
    table = pq.read_table(SESSION_PATHS[session_id])
    df = table.to_pandas()
    sub = df.iloc[switch_time_idx - half_width:switch_time_idx + half_width + 1].copy().reset_index(drop=True)
    sub['rel_time'] = np.arange(-half_width, half_width + 1)
    return sub

def rank_candidate_examples() -> pd.DataFrame:
    switch_df = pd.read_csv(SWITCH_SEGMENTS_PATH)
    rows = []
    for row in switch_df.sort_values(['segment_total_persistence_h1', 'posterior_jump_l2'], ascending=[False, False]).head(350).itertuples(index=False):
        session_id = str(row.session_id)
        if session_id not in SESSION_PATHS:
            continue
        st = int(row.switch_time_idx)
        session_df = pq.read_table(SESSION_PATHS[session_id]).to_pandas()
        if st - 35 < 0 or st + 35 >= len(session_df):
            continue
        sub = session_df.iloc[st - 30:st + 31]
        t_changes = int((sub['T'].to_numpy()[1:] != sub['T'].to_numpy()[:-1]).sum())
        c_changes = int((sub['C'].to_numpy()[1:] != sub['C'].to_numpy()[:-1]).sum())
        score = float(row.segment_total_persistence_h1) + 0.35 * float(sub['T'].nunique() + sub['C'].nunique()) + 0.08 * float(t_changes + c_changes)
        rows.append({
            'session_id': session_id,
            'switch_index': int(row.switch_index),
            'switch_time_idx': st,
            'transition_type': str(row.transition_type),
            'posterior_jump_l2': float(row.posterior_jump_l2),
            'segment_total_persistence_h1': float(row.segment_total_persistence_h1),
            'score': score,
        })
    ranked = pd.DataFrame(rows).sort_values(['score', 'segment_total_persistence_h1', 'posterior_jump_l2'], ascending=[False, False, False]).reset_index(drop=True)
    selected = []
    used_sessions = set()
    for row in ranked.itertuples(index=False):
        if row.session_id in used_sessions:
            continue
        selected.append(row._asdict())
        used_sessions.add(row.session_id)
        if len(selected) == 5:
            break
    if len(selected) < 5:
        for row in ranked.itertuples(index=False):
            keep = True
            for prior in selected:
                if row.session_id == prior['session_id'] and abs(int(row.switch_time_idx) - int(prior['switch_time_idx'])) < 25:
                    keep = False
                    break
            if not keep:
                continue
            selected.append(row._asdict())
            if len(selected) == 5:
                break
    return pd.DataFrame(selected)

def select_dims(posterior: np.ndarray, switch_time_idx: int, n_dims: int = 3) -> list[int]:
    local = posterior[switch_time_idx - 30:switch_time_idx + 31]
    return np.argsort(local.var(axis=0))[::-1][:n_dims].tolist()

def longest_h1(diagram: np.ndarray) -> np.ndarray | None:
    if diagram.size == 0:
        return None
    life = diagram[:, 1] - diagram[:, 0]
    if len(life) == 0:
        return None
    return diagram[int(np.argmax(life))]

def choose_projection_pair(coords3: np.ndarray, rel_time: np.ndarray) -> tuple[tuple[int, int], pd.DataFrame]:
    pairs = [(0, 1), (0, 2), (1, 2)]
    best_pair = (0, 1)
    best_score = -np.inf
    best_df = None
    for pair in pairs:
        df = pd.DataFrame({'rel_time': rel_time, 'x': coords3[:, pair[0]], 'y': coords3[:, pair[1]]})
        peri = df[(df['rel_time'] >= -8) & (df['rel_time'] <= 8)][['x', 'y']].to_numpy()
        post = df[(df['rel_time'] >= 8) & (df['rel_time'] <= 25)][['x', 'y']].to_numpy()
        def tort(pts):
            if len(pts) < 2:
                return 0.0
            path = float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum())
            disp = float(np.linalg.norm(pts[-1] - pts[0]))
            return path / (disp + 1e-9)
        score = tort(peri) - 0.35 * tort(post)
        if score > best_score:
            best_score = score
            best_pair = pair
            best_df = df.copy()
    return best_pair, best_df

examples = rank_candidate_examples()
records = []
prepared = []
for ex in examples.to_dict(orient='records'):
    sid = ex['session_id']
    st = int(ex['switch_time_idx'])
    affect = load_session_window(sid, st)
    posterior = posterior_matrix(MODEL, SESSION_PATHS[sid])
    dims = select_dims(posterior, st, n_dims=3)
    embedded, centers = build_delay_embedding(posterior[:, dims], m=DELAY_M, tau=DELAY_TAU)
    mask = (centers >= st - 30) & (centers <= st + 25)
    coords3 = PCA(n_components=3).fit_transform(embedded[mask])
    rel_time = centers[mask] - st
    pair, traj_df = choose_projection_pair(coords3, rel_time)
    traj_df = traj_df.rename(columns={'x': 'pcx', 'y': 'pcy'})
    traj_df['window_example'] = f"{sid} | sw {int(ex['switch_index'])}"
    peri_mask = (centers >= st - 8) & (centers <= st + 8)
    peri_diag = np.asarray(ripser(embedded[peri_mask], maxdim=1)['dgms'][1], dtype=float)
    peri_diag = peri_diag[np.isfinite(peri_diag).all(axis=1)] if peri_diag.size else np.empty((0, 2), dtype=float)
    longest = longest_h1(peri_diag)
    ex['projection_pair'] = f'PC{pair[0]+1}-PC{pair[1]+1}'
    ex['projection_dims'] = ','.join(map(str, dims))
    ex['peri_h1_lifetime'] = float(longest[1] - longest[0]) if longest is not None else np.nan
    prepared.append({'example': ex, 'affect': affect, 'posterior': posterior, 'traj': traj_df, 'diag': peri_diag})
    records.append(ex)
summary_df = pd.DataFrame(records)
summary_df.to_csv(TOP5_SUMMARY_PATH, index=False)

raster_cmap = ListedColormap(['#8C1D18', '#7A7A7A', '#2A9D8F'])
raster_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], raster_cmap.N)
time_norm = Normalize(vmin=-30, vmax=25)
time_cmap = plt.get_cmap('viridis')
state_colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#FF9DA6', '#9D755D']

# Panel 1: rasters for top 5
fig, axes = plt.subplots(len(prepared), 1, figsize=(12.8, 1.55 * len(prepared)), sharex=True, constrained_layout=True)
if len(prepared) == 1:
    axes = [axes]
for ax, pack in zip(axes, prepared, strict=False):
    affect = pack['affect']
    raster_values = np.vstack([affect['T'].to_numpy(), affect['C'].to_numpy()])
    ax.imshow(raster_values, aspect='auto', cmap=raster_cmap, norm=raster_norm, extent=[-30.5, 30.5, 1.5, -0.5])
    ax.axvline(0, color='black', linestyle='--', linewidth=1.0)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['T', 'C'], fontsize=9)
    ax.set_title(f"{pack['example']['session_id']} | switch {int(pack['example']['switch_index'])}", loc='left', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[-1].set_xticks([-30, -20, -10, 0, 10, 20, 30])
axes[-1].set_xlabel('Time relative to switch (s)', fontsize=10)
fig.savefig(TOP5_RASTER_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(TOP5_RASTER_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 2: stacked posterior areas
fig, axes = plt.subplots(len(prepared), 1, figsize=(12.8, 1.9 * len(prepared)), sharex=True, constrained_layout=True)
if len(prepared) == 1:
    axes = [axes]
for ax, pack in zip(axes, prepared, strict=False):
    ex = pack['example']
    st = int(ex['switch_time_idx'])
    t = np.arange(-30, 31)
    pw = pack['posterior'][st - 30:st + 31]
    order = np.argsort(pw.mean(axis=0))[::-1]
    ax.stackplot(t, [pw[:, k] for k in order], colors=[state_colors[k] for k in order], linewidth=0.0, alpha=0.96)
    ax.axvline(0, color='white', linestyle='--', linewidth=1.1)
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_ylabel('Mass', fontsize=9)
    ax.set_title(f"{ex['session_id']} | switch {int(ex['switch_index'])}", loc='left', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[-1].set_xticks([-30, -20, -10, 0, 10, 20, 30])
axes[-1].set_xlabel('Time relative to switch (s)', fontsize=10)
legend_handles = [plt.Line2D([0], [0], color=state_colors[k], lw=6) for k in range(8)]
axes[0].legend(legend_handles, [f'State {k}' for k in range(8)], ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(1.0, 1.33), loc='upper right')
fig.savefig(TOP5_POST_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(TOP5_POST_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 3: geometry top 5 x pre/peri/post
fig, axes = plt.subplots(len(prepared), 3, figsize=(10.8, 2.15 * len(prepared)), sharex=False, sharey=False, constrained_layout=True)
if len(prepared) == 1:
    axes = np.asarray([axes])
for i, pack in enumerate(prepared):
    traj = pack['traj']
    xpad = 0.08 * max(traj['pcx'].max() - traj['pcx'].min(), 1e-6)
    ypad = 0.08 * max(traj['pcy'].max() - traj['pcy'].min(), 1e-6)
    xlim = (traj['pcx'].min() - xpad, traj['pcx'].max() + xpad)
    ylim = (traj['pcy'].min() - ypad, traj['pcy'].max() + ypad)
    for j, (label, a, b) in enumerate(GEOM_WINDOWS):
        ax = axes[i, j]
        sub = traj[(traj['rel_time'] >= a) & (traj['rel_time'] <= b)].copy()
        ax.plot(sub['pcx'], sub['pcy'], color='0.78', linewidth=1.05, zorder=1)
        ax.scatter(sub['pcx'], sub['pcy'], c=sub['rel_time'], cmap=time_cmap, norm=time_norm, s=28, edgecolor='white', linewidth=0.4, zorder=2)
        ax.scatter(sub['pcx'].iloc[0], sub['pcy'].iloc[0], s=54, facecolor='white', edgecolor='black', linewidth=1.0, zorder=3)
        ax.scatter(sub['pcx'].iloc[-1], sub['pcy'].iloc[-1], s=54, facecolor='black', edgecolor='black', linewidth=0.8, zorder=3)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect('equal', adjustable='box')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if i == 0:
            ax.set_title(label, fontsize=11)
        if j == 0:
            ax.set_ylabel(f"{pack['example']['session_id']}\nPC2", fontsize=9)
        else:
            ax.set_yticklabels([])
        ax.set_xlabel('PC1', fontsize=9)
sm = plt.cm.ScalarMappable(norm=time_norm, cmap=time_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, fraction=0.018, pad=0.01)
cbar.set_label('Time relative to switch (s)', fontsize=9)
fig.savefig(TOP5_TRAJ_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(TOP5_TRAJ_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Panel 4: peri persistence diagrams top 5
fig, axes = plt.subplots(1, len(prepared), figsize=(3.1 * len(prepared), 3.2), constrained_layout=True)
if len(prepared) == 1:
    axes = [axes]
diag_max = 1.0
for pack in prepared:
    diag = pack['diag']
    if len(diag):
        diag_max = max(diag_max, float(diag.max()) * 1.08)
for ax, pack in zip(axes, prepared, strict=False):
    diag = pack['diag']
    if len(diag):
        ax.scatter(diag[:, 0], diag[:, 1], color='#1f77b4', s=40, alpha=0.85)
        longest = longest_h1(diag)
        if longest is not None:
            ax.scatter([longest[0]], [longest[1]], color='#D55E00', s=64, zorder=3)
    ax.plot([0, diag_max], [0, diag_max], linestyle='--', color='0.6', linewidth=1.0)
    ax.set_xlim(0, diag_max)
    ax.set_ylim(0, diag_max)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f"{pack['example']['session_id']}\nsw {int(pack['example']['switch_index'])}", fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('Birth', fontsize=9)
axes[0].set_ylabel('Death', fontsize=9)
fig.savefig(TOP5_PD_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(TOP5_PD_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print('Saved:', TOP5_RASTER_SVG)
print('Saved:', TOP5_POST_SVG)
print('Saved:', TOP5_TRAJ_SVG)
print('Saved:', TOP5_PD_SVG)
display(summary_df)


In [ ]:
from ripser import ripser
from sklearn.decomposition import PCA

METHODS_PREFIX = 'topo_transition_methods_illustration'
METHODS_SVG = FIG_DIR / f'{METHODS_PREFIX}.svg'
METHODS_PNG = FIG_DIR / f'{METHODS_PREFIX}.png'
METHODS_SUMMARY = TAB_DIR / f'{METHODS_PREFIX}_summary.csv'

METHODS_EXAMPLE = {
    'session_id': '154_A',
    'switch_index': 406,
    'switch_time_idx': 1278,
}
METHODS_WINDOW = (-8, 8)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
plt.rcParams['svg.fonttype'] = 'path'

def choose_methods_projection(coords3: np.ndarray, rel_time: np.ndarray) -> tuple[tuple[int, int], np.ndarray]:
    best_pair = (0, 1)
    best_score = -np.inf
    best_coords = coords3[:, :2]
    for pair in [(0, 1), (0, 2), (1, 2)]:
        pts = coords3[:, [pair[0], pair[1]]]
        path = float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum()) if len(pts) > 1 else 0.0
        disp = float(np.linalg.norm(pts[-1] - pts[0])) if len(pts) > 1 else 0.0
        score = path / (disp + 1e-9)
        if score > best_score:
            best_score = score
            best_pair = pair
            best_coords = pts
    return best_pair, best_coords

def longest_h1_methods(diagram: np.ndarray) -> np.ndarray | None:
    if diagram.size == 0:
        return None
    life = diagram[:, 1] - diagram[:, 0]
    return diagram[int(np.argmax(life))] if len(life) else None

example = METHODS_EXAMPLE.copy()
sid = example['session_id']
st = int(example['switch_time_idx'])
posterior = posterior_matrix(MODEL, SESSION_PATHS[sid])
posterior_window = posterior[st - 30:st + 31]
state_order = np.argsort(posterior_window.mean(axis=0))[::-1]

dims = np.argsort(posterior[st - 30:st + 31].var(axis=0))[::-1][:3].tolist()
embedded, centers = build_delay_embedding(posterior[:, dims], m=DELAY_M, tau=DELAY_TAU)
peri_mask = (centers >= st + METHODS_WINDOW[0]) & (centers <= st + METHODS_WINDOW[1])
peri_points = embedded[peri_mask]
peri_rel_time = centers[peri_mask] - st
coords3 = PCA(n_components=3).fit_transform(peri_points)
proj_pair, proj2 = choose_methods_projection(coords3, peri_rel_time)
diagram = np.asarray(ripser(peri_points, maxdim=1)['dgms'][1], dtype=float)
diagram = diagram[np.isfinite(diagram).all(axis=1)] if diagram.size else np.empty((0, 2), dtype=float)
main_h1 = longest_h1_methods(diagram)

summary_df = pd.DataFrame([
    {
        'session_id': sid,
        'switch_index': int(example['switch_index']),
        'switch_time_idx': st,
        'projection_dims': ','.join(map(str, dims)),
        'projection_pair': f'PC{proj_pair[0] + 1}-PC{proj_pair[1] + 1}',
        'peri_points': int(len(peri_points)),
        'h1_birth': float(main_h1[0]) if main_h1 is not None else np.nan,
        'h1_death': float(main_h1[1]) if main_h1 is not None else np.nan,
        'h1_lifetime': float(main_h1[1] - main_h1[0]) if main_h1 is not None else np.nan,
    }
])
summary_df.to_csv(METHODS_SUMMARY, index=False)

time_norm = Normalize(vmin=METHODS_WINDOW[0], vmax=METHODS_WINDOW[1])
time_cmap = plt.get_cmap('viridis')
xpad = 0.1 * max(proj2[:, 0].max() - proj2[:, 0].min(), 1e-6)
ypad = 0.1 * max(proj2[:, 1].max() - proj2[:, 1].min(), 1e-6)
diag_max = max(1.0, float(diagram.max()) * 1.08) if len(diagram) else 1.0

fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.9), width_ratios=[1.7, 1.0, 1.0], constrained_layout=True)

ax = axes[0]
ax.stackplot(
    np.arange(-30, 31),
    [posterior_window[:, k] for k in state_order],
    colors=['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#FF9DA6', '#9D755D'],
    linewidth=0.0,
    alpha=0.96,
)
ax.axvspan(METHODS_WINDOW[0], METHODS_WINDOW[1], color='0.85', alpha=0.35, zorder=0)
ax.axvline(0, color='white', linestyle='--', linewidth=1.2)
ax.set_xlim(-30, 30)
ax.set_ylim(0, 1)
ax.set_xlabel('Time relative to switch (s)', fontsize=10)
ax.set_ylabel('Posterior mass', fontsize=10)
ax.set_title('Posterior Redistribution', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
ax.plot(proj2[:, 0], proj2[:, 1], color='0.78', linewidth=1.2, zorder=1)
ax.scatter(proj2[:, 0], proj2[:, 1], c=peri_rel_time, cmap=time_cmap, norm=time_norm, s=42, edgecolor='white', linewidth=0.5, zorder=2)
ax.scatter(proj2[0, 0], proj2[0, 1], s=76, facecolor='white', edgecolor='black', linewidth=1.0, zorder=3)
ax.scatter(proj2[-1, 0], proj2[-1, 1], s=76, facecolor='black', edgecolor='black', linewidth=0.8, zorder=3)
ax.set_xlabel('PC1', fontsize=10)
ax.set_ylabel('PC2', fontsize=10)
ax.set_title('Delay-Embedded Trajectory', fontsize=13)
ax.set_aspect('equal', adjustable='box')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
sm = plt.cm.ScalarMappable(norm=time_norm, cmap=time_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.03)
cbar.set_label('Time within peri-switch window (s)', fontsize=9)

ax = axes[2]
if len(diagram):
    ax.scatter(diagram[:, 0], diagram[:, 1], color='#1f77b4', s=48, alpha=0.88)
if main_h1 is not None:
    ax.scatter([main_h1[0]], [main_h1[1]], color='#D55E00', s=76, zorder=3)
    ax.annotate(
        'H1 loop',
        xy=(main_h1[0], main_h1[1]),
        xytext=(main_h1[0] + 0.08, main_h1[1] + 0.1),
        arrowprops=dict(arrowstyle='->', lw=1.0, color='#D55E00'),
        fontsize=9,
        color='#7A3B00',
    )
ax.plot([0, diag_max], [0, diag_max], linestyle='--', color='0.6', linewidth=1.0)
ax.set_xlim(0, diag_max)
ax.set_ylim(0, diag_max)
ax.set_xlabel('Birth', fontsize=10)
ax.set_ylabel('Death', fontsize=10)
ax.set_title('Persistence Diagram', fontsize=13)
ax.set_aspect('equal', adjustable='box')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.savefig(METHODS_SVG, format='svg', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(METHODS_PNG, format='png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Saved SVG to', METHODS_SVG)
print('Saved PNG to', METHODS_PNG)
display(summary_df)
